In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

project_path = "/content/drive/MyDrive/Legal_Project"

print("=" * 80)
print("LEGAL PROJECT DIRECTORY")
print("=" * 80)

if os.path.exists(project_path):
    print("Project folder found successfully!\n")
    print("Contents:")
    print(os.listdir(project_path))
else:
    print("Legal_Project folder was NOT found.")

LEGAL PROJECT DIRECTORY
Project folder found successfully!

Contents:
['Court_PDFs', 'Generated_Datasets', 'Dataset', 'Models']


In [ ]:
dataset_folder = "/content/drive/MyDrive/Legal_Project/Dataset"

print("=" * 80)
print("DATASET DIRECTORY")
print("=" * 80)

if os.path.exists(dataset_folder):
    print("Dataset folder found successfully!\n")
    print("Files:")

    for file in os.listdir(dataset_folder):
        print(" -", file)
else:
    print("Dataset folder was NOT found.")

DATASET DIRECTORY
Dataset folder found successfully!

Files:
 - case_files_total.csv


In [ ]:
import pandas as pd

dataset_path = "/content/drive/MyDrive/Legal_Project/Dataset/case_files_total.csv"

df = pd.read_csv(dataset_path)

print("=" * 80)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 80)

print(f"Dataset Shape : {df.shape}")

print("\nColumns:")
print(df.columns.tolist())

DATASET LOADED SUCCESSFULLY
Dataset Shape : (53446, 10)

Columns:
['Unnamed: 0', 'name', 'case_category', 'case_type', 'case_info', 'judgement', 'tokens', 'sentences', 'label', 'proof_sentence']


In [ ]:
import os

generated_path = "/content/drive/MyDrive/Legal_Project/Generated_Datasets"

print("=" * 80)
print("GENERATED DATASETS DIRECTORY")
print("=" * 80)

if os.path.exists(generated_path):
    print("Folder found successfully!\n")
    print("Files:")

    for file in os.listdir(generated_path):
        print(" -", file)
else:
    print("Generated_Datasets folder was NOT found.")

GENERATED DATASETS DIRECTORY
Folder found successfully!

Files:
 - sample_dataset.csv
 - final_dataset.csv
 - sample_dataset.gsheet
 - final_dataset.gsheet


In [ ]:
final_dataset_path = "/content/drive/MyDrive/Legal_Project/Generated_Datasets/final_dataset.csv"

final_df = pd.read_csv(final_dataset_path)

print("=" * 80)
print("FINAL DATASET INSPECTION")
print("=" * 80)

print(f"Shape : {final_df.shape}")

print("\nColumns:")
print(final_df.columns.tolist())

FINAL DATASET INSPECTION
Shape : (265, 13)

Columns:
['file_name', 'court_name', 'citation_number', 'case_number', 'case_type', 'date', 'judge_name', 'petitioner', 'respondent', 'judgement_text', 'final_order', 'proof_sentence', 'label']


In [ ]:
# ============================================================
# STEP 2: Recreate the Binary Modeling Dataset
# ============================================================

# Work on a copy so the original 53,446-row dataset remains untouched
model_df = df.copy()

# ------------------------------------------------------------
# Remove rows where judgement or label is missing
# ------------------------------------------------------------

model_df = model_df.dropna(subset=["judgement", "label"])

# Reset index
model_df.reset_index(drop=True, inplace=True)

print("=" * 80)
print("AFTER REMOVING MISSING JUDGMENT/LABEL")
print("=" * 80)

print(f"Rows Remaining : {len(model_df):,}")

# ------------------------------------------------------------
# Keep only Accepted and Rejected cases
# ------------------------------------------------------------

model_df = model_df[
    model_df["label"].isin(["Accepted", "Rejected"])
].copy()

# Encode labels
label_mapping = {
    "Rejected": 0,
    "Accepted": 1
}

model_df["label"] = model_df["label"].map(label_mapping)

# Reset index
model_df.reset_index(drop=True, inplace=True)

print("\n" + "=" * 80)
print("BINARY DATASET CREATED")
print("=" * 80)

print(f"Total Samples : {len(model_df):,}")

print("\nLabel Distribution:")
print(model_df["label"].value_counts().sort_index())

print("\nLabel Meaning:")
print("0 = Rejected")
print("1 = Accepted")

AFTER REMOVING MISSING JUDGMENT/LABEL
Rows Remaining : 50,143

BINARY DATASET CREATED
Total Samples : 42,342

Label Distribution:
label
0    22510
1    19832
Name: count, dtype: int64

Label Meaning:
0 = Rejected
1 = Accepted


In [ ]:
# ============================================================
# STEP 3: Remove Unnecessary Columns
# ============================================================

columns_to_drop = [
    "Unnamed: 0",
    "name",
    "tokens",
    "sentences",
    "proof_sentence"
]

model_df.drop(columns=columns_to_drop, inplace=True)

print("=" * 80)
print("UNNECESSARY COLUMNS REMOVED")
print("=" * 80)

print("Remaining Columns:")
print(model_df.columns.tolist())

print("\nDataset Shape:")
print(model_df.shape)

UNNECESSARY COLUMNS REMOVED
Remaining Columns:
['case_category', 'case_type', 'case_info', 'judgement', 'label']

Dataset Shape:
(42342, 5)


In [ ]:
# ============================================================
# STEP 4: Create Text Statistics
# ============================================================

# Character count
model_df["char_count"] = model_df["judgement"].str.len()

# Word count
model_df["word_count"] = model_df["judgement"].str.split().str.len()

# Sentence count (approximate)
model_df["sentence_count"] = model_df["judgement"].str.count(r"\.") + 1

print("=" * 80)
print("TEXT STATISTICS CREATED SUCCESSFULLY")
print("=" * 80)

display(
    model_df[["char_count", "word_count", "sentence_count"]].head()
)

print("\nDataset Shape:", model_df.shape)

TEXT STATISTICS CREATED SUCCESSFULLY


,char_count,word_count,sentence_count
0,32765,5762,207
1,3033,537,31
2,14385,2510,104
3,14772,2643,163
4,13008,2383,178



Dataset Shape: (42342, 8)


In [ ]:
# ============================================================
# STEP 5: Remove Extremely Short Judgments
# ============================================================

before_rows = len(model_df)

# Keep judgments having at least 50 words
model_df = model_df[model_df["word_count"] >= 50].copy()

# Reset index
model_df.reset_index(drop=True, inplace=True)

print("=" * 80)
print("DATASET AFTER REMOVING EXTREMELY SHORT JUDGMENTS")
print("=" * 80)

print(f"Rows Before Cleaning : {before_rows:,}")
print(f"Rows After Cleaning  : {len(model_df):,}")
print(f"Rows Removed         : {before_rows - len(model_df):,}")

print("\nDataset Shape:", model_df.shape)

DATASET AFTER REMOVING EXTREMELY SHORT JUDGMENTS
Rows Before Cleaning : 42,342
Rows After Cleaning  : 42,169
Rows Removed         : 173

Dataset Shape: (42169, 8)


In [ ]:
# ============================================================
# STEP 6: Initialize Text Preprocessing Resources
# ============================================================

import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download required NLTK resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")
nltk.download("punkt_tab")

print("=" * 80)
print("NLTK RESOURCES READY")
print("=" * 80)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


NLTK RESOURCES READY


In [ ]:
# ============================================================
# STEP 7: Create Text Preprocessing Function
# ============================================================

# Initialize stopwords
stop_words = set(stopwords.words("english"))

# Preserve important legal negation words
important_words = {"not", "no", "nor"}

# Remove negation words from stopword list
stop_words = stop_words - important_words

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
    """
    Preprocess legal judgment text.

    Steps:
    1. Handle missing values
    2. Convert text to lowercase
    3. Remove special characters
    4. Tokenize
    5. Remove stopwords
    6. Preserve important negation words
    7. Remove very short words
    8. Lemmatize
    9. Join tokens back into a string
    """

    # Handle missing values
    if pd.isna(text):
        return ""

    # Convert to lowercase
    text = text.lower()

    # Keep alphanumeric characters and spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Tokenize
    words = word_tokenize(text)

    # Remove stopwords and very short words
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words and len(word) > 2
    ]

    # Join words back into a single string
    return " ".join(words)


print("=" * 80)
print("TEXT PREPROCESSING FUNCTION CREATED")
print("=" * 80)

print("Negation words preserved:", important_words)

TEXT PREPROCESSING FUNCTION CREATED
Negation words preserved: {'nor', 'no', 'not'}


In [ ]:
# ============================================================
# STEP 8: Test Text Preprocessing Function
# ============================================================

sample_index = 0

original_text = model_df.loc[sample_index, "judgement"]
cleaned_text = preprocess_text(original_text)

print("=" * 80)
print("ORIGINAL JUDGMENT (First 1000 Characters)")
print("=" * 80)
print(original_text[:1000])

print("\n\n")

print("=" * 80)
print("PREPROCESSED JUDGMENT (First 1000 Characters)")
print("=" * 80)
print(cleaned_text[:1000])

ORIGINAL JUDGMENT (First 1000 Characters)
thus spake benjamin franklin in his letter of november 13, 1789 to jean baptiste leroy.
to tax the dead is a contradiction in terms.
tax laws are made by the living to tax the living.
what survives the dead person is what is left behind in the form of such persons property.
this appeal raises questions as to whether the dead persons property, in the form of his or her estate, can be taxed without the necessary machinery provisions in a tax statute.
the precise question that arises in the present case is whether an assessment proceeding under the central excises and salt act, 1944, can continue against the legal representatives estate of a sole proprietor manufacturer after he is dead.
the facts of the case are as follows.
2. one shri george varghese was the sole proprietor of kerala tyre and rubber company limited.
by october 1985, this proprietary concern had stopped manufacture and production of tread rubber.
by a show cause notice dated 12.6

In [ ]:
# ============================================================
# STEP 9: Apply Preprocessing to All Judgments
# ============================================================

print("Preprocessing all legal judgments...")

# Create cleaned judgment column
model_df["cleaned_judgement"] = model_df["judgement"].apply(preprocess_text)

print("\nPreprocessing completed successfully!")

print("\nDataset Shape:", model_df.shape)

print("\nColumns:")
print(model_df.columns.tolist())

Preprocessing all legal judgments...

Preprocessing completed successfully!

Dataset Shape: (42169, 9)

Columns:
['case_category', 'case_type', 'case_info', 'judgement', 'label', 'char_count', 'word_count', 'sentence_count', 'cleaned_judgement']


In [ ]:
# ============================================================
# STEP 10: Save Preprocessed Modeling Dataset
# ============================================================

preprocessed_path = (
    "/content/drive/MyDrive/Legal_Project/Dataset/"
    "case_files_preprocessed.csv"
)

model_df.to_csv(preprocessed_path, index=False)

print("=" * 80)
print("PREPROCESSED DATASET SAVED SUCCESSFULLY")
print("=" * 80)

print(f"Location : {preprocessed_path}")
print(f"Shape    : {model_df.shape}")

PREPROCESSED DATASET SAVED SUCCESSFULLY
Location : /content/drive/MyDrive/Legal_Project/Dataset/case_files_preprocessed.csv
Shape    : (42169, 9)


In [ ]:
# ============================================================
# STEP 11: Inspect Article References in Real Judgments
# ============================================================

import re

# Find occurrences of "article" / "articles" in the original judgments
article_pattern = re.compile(
    r"\barticles?\b",
    re.IGNORECASE
)

article_rows = model_df[
    model_df["judgement"].str.contains(
        article_pattern,
        na=False
    )
].copy()

print("=" * 80)
print("ARTICLE REFERENCE INSPECTION")
print("=" * 80)

print(f"Total Judgments        : {len(model_df):,}")
print(f"Judgments mentioning 'Article(s)' : {len(article_rows):,}")

print(
    f"Percentage             : "
    f"{len(article_rows) / len(model_df) * 100:.2f}%"
)

ARTICLE REFERENCE INSPECTION
Total Judgments        : 42,169
Judgments mentioning 'Article(s)' : 14,852
Percentage             : 35.22%


In [ ]:
# ============================================================
# STEP 12: Inspect Real Article References
# ============================================================

def get_article_context(text, window=120):
    """
    Extract small text snippets around occurrences
    of the word 'Article' or 'Articles'.
    """

    matches = list(
        re.finditer(r"\barticles?\b", text, re.IGNORECASE)
    )

    snippets = []

    for match in matches[:3]:
        start = max(0, match.start() - window)
        end = min(len(text), match.end() + window)

        snippet = text[start:end].replace("\n", " ")
        snippets.append(snippet)

    return snippets


# Inspect first 20 judgments containing "Article"
sample_article_rows = article_rows.head(20).copy()

print("=" * 80)
print("SAMPLE ARTICLE REFERENCES FROM REAL JUDGMENTS")
print("=" * 80)

for i, (_, row) in enumerate(sample_article_rows.iterrows(), start=1):

    print(f"\n{'-' * 80}")
    print(f"Sample {i}")
    print(f"{'-' * 80}")

    snippets = get_article_context(row["judgement"])

    for j, snippet in enumerate(snippets, start=1):
        print(f"\nContext {j}:")
        print(snippet)

SAMPLE ARTICLE REFERENCES FROM REAL JUDGMENTS

--------------------------------------------------------------------------------
Sample 1
--------------------------------------------------------------------------------

Context 1:
to pass an order on the maintainability of the show cause notice alone, the legal heirs approached the high court under article 226 of the constitution by filing a writ petition in january, 1990. the learned single judge of the high court quashed 

--------------------------------------------------------------------------------
Sample 2
--------------------------------------------------------------------------------

Context 1:
ing about an action in tort. in the alternative it was submitted that this court, in exercise of its jurisdiction under article 136 of the constitution, should not interfere in the present case. there are only three provisions of the esi which are

--------------------------------------------------------------------------------
Sample 3

In [ ]:
# ============================================================
# STEP 13: Analyze Patterns Following "Article"
# ============================================================

# Extract the text immediately following "Article" / "Articles"
pattern_samples = []

for text in article_rows["judgement"]:
    matches = re.finditer(
        r"\barticles?\b.{0,40}",
        text,
        re.IGNORECASE
    )

    for match in matches:
        snippet = match.group(0).replace("\n", " ")
        pattern_samples.append(snippet)

# Remove duplicates and display a sample
unique_patterns = list(dict.fromkeys(pattern_samples))

print("=" * 80)
print("COMMON PATTERNS FOLLOWING 'ARTICLE'")
print("=" * 80)

print(f"Total extracted patterns : {len(pattern_samples):,}")
print(f"Unique patterns          : {len(unique_patterns):,}")

print("\nFirst 100 unique patterns:\n")

for i, pattern in enumerate(unique_patterns[:100], start=1):
    print(f"{i:3}. {pattern}")

COMMON PATTERNS FOLLOWING 'ARTICLE'
Total extracted patterns : 75,387
Unique patterns          : 41,538

First 100 unique patterns:

  1. article 226 of the constitution by filing a wri
  2. article 136 of the constitution, should not int
  3. article 136 of the constitution and has preferr
  4. article 136 of the constitution with the conclu
  5. article 14 and right to property under article 
  6. article 16  4a  of the constitution of india is
  7. article 16 4  of the constitution of india is c
  8. article 16 by constitution  seventy-seventh ame
  9. article 16 of the constitution.
 10. article 16 of the constitution of india read as
 11. article shall prevent the state from making any
 12. article 16  4  of the constitution of india ena
 13. article 16 is that the state is now empowered t
 14. article 16 of the constitution is only an enabl
 15. article 16 was further amended enabling the sta
 16. article 16 of the constitution of india.
 17. articles 14 and 16 of the constitutio

In [ ]:
# ============================================================
# STEP 14: Explore Candidate Article Numbers
# ============================================================

# Candidate pattern:
# Article / Articles followed by a number or number-letter combination
candidate_pattern = re.compile(
    r"\barticles?\s*[-:]?\s*(\d{1,3}[a-zA-Z]?)",
    re.IGNORECASE
)

candidate_articles = []

for text in model_df["judgement"]:
    matches = candidate_pattern.findall(text)
    candidate_articles.extend(matches)

# Normalize
candidate_articles = [
    article.upper()
    for article in candidate_articles
]

# Count frequencies
article_counts = pd.Series(candidate_articles).value_counts()

print("=" * 80)
print("CANDIDATE ARTICLE NUMBER ANALYSIS")
print("=" * 80)

print(f"Total candidate references : {len(candidate_articles):,}")
print(f"Unique candidate values    : {len(article_counts):,}")

print("\nTop 30 Candidate Article Values:")
print(article_counts.head(30))

CANDIDATE ARTICLE NUMBER ANALYSIS
Total candidate references : 57,065
Unique candidate values    : 557

Top 30 Candidate Article Values:
226    6862
14     6804
136    3859
19     2261
32     1756
311    1643
227    1529
16     1472
21     1427
309    1343
142    1223
31      898
22      824
12      767
20      664
133     631
15      626
286     567
301     511
30      501
137     452
254     441
366     434
39      402
134     381
166     368
141     351
233     321
304     318
72      312
Name: count, dtype: int64


In [ ]:
# ============================================================
# STEP 15: Inspect Unusual Candidate Article Values
# ============================================================

print("=" * 80)
print("CANDIDATE ARTICLE VALUES - LEAST FREQUENT")
print("=" * 80)

print("\nCandidates occurring only once:")
print(article_counts[article_counts == 1].head(100))

print("\n" + "=" * 80)
print("HIGHEST NUMERICAL ARTICLE CANDIDATES")
print("=" * 80)

# Convert numeric-looking values to integers for inspection
numeric_candidates = []

for value in article_counts.index:
    match = re.match(r"^\d+$", str(value))

    if match:
        numeric_candidates.append(int(value))

print(
    sorted(numeric_candidates, reverse=True)[:100]
)

CANDIDATE ARTICLE VALUES - LEAST FREQUENT

Candidates occurring only once:
342O    1
232     1
274     1
363A    1
237     1
       ..
135A    1
931     1
14T     1
72O     1
11A     1
Name: count, Length: 91, dtype: int64

HIGHEST NUMERICAL ARTICLE CANDIDATES
[998, 934, 931, 926, 842, 830, 811, 810, 809, 790, 776, 761, 750, 722, 715, 686, 677, 675, 669, 668, 660, 637, 626, 606, 586, 567, 565, 563, 540, 535, 529, 528, 526, 521, 520, 516, 515, 512, 511, 510, 505, 501, 474, 470, 466, 465, 462, 459, 456, 438, 429, 428, 420, 395, 394, 392, 391, 389, 386, 385, 384, 382, 379, 377, 374, 372, 371, 370, 369, 368, 367, 366, 364, 363, 362, 361, 360, 359, 358, 357, 356, 355, 353, 352, 351, 350, 349, 348, 347, 346, 345, 344, 343, 342, 341, 340, 339, 338, 337, 335]


In [ ]:
# ============================================================
# STEP 16: Inspect Suspicious High-Number Article Candidates
# ============================================================

suspicious_values = ["998", "934", "931", "926", "842", "830", "686"]

print("=" * 80)
print("INSPECTING SUSPICIOUS ARTICLE CANDIDATES")
print("=" * 80)

for value in suspicious_values:

    print(f"\n{'-' * 80}")
    print(f"Candidate Article: {value}")
    print(f"{'-' * 80}")

    pattern = re.compile(
        rf"\barticles?\s*[-:]?\s*{re.escape(value)}\b",
        re.IGNORECASE
    )

    found = False

    for text in model_df["judgement"]:
        match = pattern.search(text)

        if match:
            start = max(0, match.start() - 150)
            end = min(len(text), match.end() + 150)

            print(text[start:end].replace("\n", " "))
            found = True
            break

    if not found:
        print("No example found.")

INSPECTING SUSPICIOUS ARTICLE CANDIDATES

--------------------------------------------------------------------------------
Candidate Article: 998
--------------------------------------------------------------------------------
egulate trade on public streets and not altogether prevent the same. our attention was drawn to halsburys laws of england, vol. -33   third edition , article 998 at page 586 headed regulation of street trading. the learned author thus summarised the law in england- subject to certain exceptions it is unlawful 

--------------------------------------------------------------------------------
Candidate Article: 934
--------------------------------------------------------------------------------
ent on the ground that the respondent shifted his residence from ceylon to uganda and was, therefore, entitled to benefit under the second proviso to article 934 of the civil service regulations. the second proviso to article 934 was as follows- provided that save where a pe

In [ ]:
# ============================================================
# STEP 17: Create Constitutional Article Reference Set
# ============================================================

# Core Constitutional Article numbers
valid_articles = {
    str(i) for i in range(1, 396)
}

# Lettered / inserted Constitutional Articles
inserted_articles = {
    "2A",
    "21A",
    "31A",
    "31B",
    "31C",
    "39A",
    "43A",
    "48A",
    "51A",
    "239A",
    "239AA",
    "243A",
    "243B",
    "243C",
    "243D",
    "243E",
    "243F",
    "243G",
    "243H",
    "243I",
    "243J",
    "243K",
    "243L",
    "243M",
    "243N",
    "243O",
    "243P",
    "243Q",
    "243R",
    "243S",
    "243T",
    "243U",
    "243V",
    "243W",
    "243X",
    "243Y",
    "243Z",
    "243ZA",
    "243ZB",
    "243ZC",
    "243ZD",
    "243ZE",
    "243ZF",
    "243ZG",
    "244A",
    "300A",
    "323A",
    "323B",
    "330A",
    "332A",
    "334A",
    "342A",
    "371A",
    "371B",
    "371C",
    "371D",
    "371E",
    "371F",
    "371G",
    "371H",
    "371I",
    "371J",
    "372A",
    "378A",
    "394A"
}

valid_articles.update(inserted_articles)

print("=" * 80)
print("CONSTITUTIONAL ARTICLE REFERENCE SET")
print("=" * 80)

print(f"Total reference identifiers : {len(valid_articles):,}")

print("\nSample identifiers:")
print(sorted(valid_articles, key=lambda x: (int(re.match(r'\d+', x).group()), x))[:50])

print("\nImportant inserted Articles:")
print(sorted(inserted_articles))

CONSTITUTIONAL ARTICLE REFERENCE SET
Total reference identifiers : 460

Sample identifiers:
['1', '2', '2A', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '21A', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '31A', '31B', '31C', '32', '33', '34', '35', '36', '37', '38', '39', '39A', '40', '41', '42', '43', '43A']

Important inserted Articles:
['21A', '239A', '239AA', '243A', '243B', '243C', '243D', '243E', '243F', '243G', '243H', '243I', '243J', '243K', '243L', '243M', '243N', '243O', '243P', '243Q', '243R', '243S', '243T', '243U', '243V', '243W', '243X', '243Y', '243Z', '243ZA', '243ZB', '243ZC', '243ZD', '243ZE', '243ZF', '243ZG', '244A', '2A', '300A', '31A', '31B', '31C', '323A', '323B', '330A', '332A', '334A', '342A', '371A', '371B', '371C', '371D', '371E', '371F', '371G', '371H', '371I', '371J', '372A', '378A', '394A', '39A', '43A', '48A', '51A']


In [ ]:
# ============================================================
# STEP 18: Validate Candidate Articles Against Reference Set
# ============================================================

# Count candidate values that are present in the reference set
recognized_candidates = [
    value
    for value in candidate_articles
    if value in valid_articles
]

unrecognized_candidates = [
    value
    for value in candidate_articles
    if value not in valid_articles
]

print("=" * 80)
print("ARTICLE CANDIDATE VALIDATION")
print("=" * 80)

print(f"Total Candidate References : {len(candidate_articles):,}")
print(f"Recognized References     : {len(recognized_candidates):,}")
print(f"Unrecognized References   : {len(unrecognized_candidates):,}")

print(
    f"\nRecognized Percentage      : "
    f"{len(recognized_candidates) / len(candidate_articles) * 100:.2f}%"
)

print(
    f"Unrecognized Percentage    : "
    f"{len(unrecognized_candidates) / len(candidate_articles) * 100:.2f}%"
)

print("\n" + "=" * 80)
print("MOST COMMON RECOGNIZED ARTICLES")
print("=" * 80)

recognized_counts = pd.Series(recognized_candidates).value_counts()

print(recognized_counts.head(30))

ARTICLE CANDIDATE VALIDATION
Total Candidate References : 57,065
Recognized References     : 56,504
Unrecognized References   : 561

Recognized Percentage      : 99.02%
Unrecognized Percentage    : 0.98%

MOST COMMON RECOGNIZED ARTICLES
226    6862
14     6804
136    3859
19     2261
32     1756
311    1643
227    1529
16     1472
21     1427
309    1343
142    1223
31      898
22      824
12      767
20      664
133     631
15      626
286     567
301     511
30      501
137     452
254     441
366     434
39      402
134     381
166     368
141     351
233     321
304     318
72      312
Name: count, dtype: int64


In [ ]:
# ============================================================
# STEP 19: Analyze Multiple Article References
# ============================================================

multi_article_pattern = re.compile(
    r"\barticles?\b\s+"
    r"\d{1,3}[a-zA-Z]?"
    r"(?:\s*(?:,|and)\s*\d{1,3}[a-zA-Z]?)+",
    re.IGNORECASE
)

multi_article_matches = []

for text in model_df["judgement"]:
    matches = multi_article_pattern.findall(text)
    multi_article_matches.extend(matches)

print("=" * 80)
print("MULTIPLE ARTICLE REFERENCE ANALYSIS")
print("=" * 80)

print(f"Multi-Article Expressions Found : {len(multi_article_matches):,}")

print("\nFirst 50 examples:\n")

for i, match in enumerate(
    list(dict.fromkeys(multi_article_matches))[:50],
    start=1
):
    print(f"{i:3}. {match}")

MULTIPLE ARTICLE REFERENCE ANALYSIS
Multi-Article Expressions Found : 4,836

First 50 examples:

  1. articles 14 and 16
  2. articles 14, 15 and 16
  3. articles 14 and 19
  4. articles 72 and 161
  5. articles 20 and 21
  6. articles 226 and 227
  7. articles 38, 39
  8. articles 249, 250, 252 and 357
  9. article 285 and 289
 10. articles 64 and 65
 11. article 300a and 14
 12. article 14 and 16
 13. articles 81, 82 and 170
 14. articles 14,15 and 21
 15. articles 14 and 21
 16. articles 278, 279 and 295
 17. articles 278, 291, 295 and 306
 18. article 226 and 226
 19. articles 14, 15, 16, 21, 38, 39 and 46
 20. articles 14, 15
 21. articles 341 and 342
 22. articles 38 and 39
 23. articles 129 and 142
 24. articles 8 and 9
 25. articles 132, 133, 134 and 134a
 26. articles 226, 227 and 32
 27. articles 323a and 323b
 28. articles 141 and 142
 29. articles 14, 19 and 21
 30. articles 134, 134a, 134b, 142
 31. articles 137 and 145
 32. articles 25 and 26
 33. articles 26,27 and 28
 3

In [ ]:
# ============================================================
# STEP 20: Create Case Article Extraction Function
# ============================================================

# Pattern for Article references
article_reference_pattern = re.compile(
    r"""
    \b
    (?:articles?|arts?)       # Article / Articles / Art / Arts
    \.?
    \s*
    [-:]?
    \s*
    (
        \d{1,3}[a-zA-Z]?
        (?:
            \s*
            (?:,|and)
            \s*
            \d{1,3}[a-zA-Z]?
        )*
    )
    """,
    re.IGNORECASE | re.VERBOSE
)


def normalize_article_identifier(value):
    """
    Normalize an extracted Article identifier.
    Example:
        300a -> 300A
        134a -> 134A
    """

    value = value.strip().upper()

    return value


def extract_articles(text):
    """
    Extract valid Indian Constitutional Article identifiers
    from a legal judgment.

    Returns:
        Comma-separated unique Article identifiers.
        Empty string if no valid Article is found.
    """

    if pd.isna(text) or not str(text).strip():
        return ""

    text = str(text)

    extracted_articles = []

    # Find Article / Articles expressions
    matches = article_reference_pattern.finditer(text)

    for match in matches:

        article_expression = match.group(1)

        # Extract individual Article identifiers
        candidates = re.findall(
            r"\d{1,3}[a-zA-Z]?",
            article_expression
        )

        for candidate in candidates:

            normalized = normalize_article_identifier(candidate)

            # Validate against Constitutional reference set
            if normalized in valid_articles:
                extracted_articles.append(normalized)

    # Remove duplicates while preserving order
    unique_articles = list(dict.fromkeys(extracted_articles))

    return ",".join(unique_articles)


print("=" * 80)
print("ARTICLE EXTRACTION FUNCTION CREATED")
print("=" * 80)

print("Function: extract_articles()")
print("Reference identifiers available:", len(valid_articles))

ARTICLE EXTRACTION FUNCTION CREATED
Function: extract_articles()
Reference identifiers available: 460


In [ ]:
# ============================================================
# STEP 21: Test Article Extraction Function
# ============================================================

test_cases = [
    "Article 226 of the Constitution",
    "Article 136 of the Constitution",
    "Articles 14 and 19 of the Constitution",
    "Articles 14, 15 and 16 of the Constitution",
    "Article 16 4a of the Constitution",
    "Article 20 2 of the Constitution",
    "Article 19 1 g of the Constitution",
    "Article 300 a of the Constitution",
    "Articles 134, 134a, 134b and 142",
    "Articles 323a and 323b",
    "Article-136 of the Constitution",
    "Art. 133 of the Constitution",
    "Articles into possession and sealed them",
    "articles of material described in sub-item",
    "Article shall prevent the State from making any law",
    "Article 226 and 226",
]

print("=" * 80)
print("ARTICLE EXTRACTION TEST")
print("=" * 80)

for i, text in enumerate(test_cases, start=1):
    result = extract_articles(text)

    print(f"\n{i:02}. Input  : {text}")
    print(f"    Output : {result}")

ARTICLE EXTRACTION TEST

01. Input  : Article 226 of the Constitution
    Output : 226

02. Input  : Article 136 of the Constitution
    Output : 136

03. Input  : Articles 14 and 19 of the Constitution
    Output : 14,19

04. Input  : Articles 14, 15 and 16 of the Constitution
    Output : 14,15,16

05. Input  : Article 16 4a of the Constitution
    Output : 16

06. Input  : Article 20 2 of the Constitution
    Output : 20

07. Input  : Article 19 1 g of the Constitution
    Output : 19

08. Input  : Article 300 a of the Constitution
    Output : 300

09. Input  : Articles 134, 134a, 134b and 142
    Output : 134,142

10. Input  : Articles 323a and 323b
    Output : 323A,323B

11. Input  : Article-136 of the Constitution
    Output : 136

12. Input  : Art. 133 of the Constitution
    Output : 133

13. Input  : Articles into possession and sealed them
    Output : 

14. Input  : articles of material described in sub-item
    Output : 

15. Input  : Article shall prevent the State from 

In [ ]:
# ============================================================
# STEP 22: Analyze Lettered Article Identifiers
# ============================================================

# Find Article/Articles followed by a number + letter
lettered_article_pattern = re.compile(
    r"\barticles?\s*[-:]?\s*(\d{1,3}\s*[a-zA-Z])\b",
    re.IGNORECASE
)

lettered_candidates = []

for text in model_df["judgement"]:
    matches = lettered_article_pattern.findall(text)

    for match in matches:
        # Remove spaces and normalize case
        normalized = re.sub(r"\s+", "", match).upper()
        lettered_candidates.append(normalized)

lettered_counts = pd.Series(lettered_candidates).value_counts()

print("=" * 80)
print("LETTERED ARTICLE IDENTIFIERS FOUND IN DATASET")
print("=" * 80)

print(f"Total lettered references : {len(lettered_candidates):,}")
print(f"Unique identifiers        : {len(lettered_counts):,}")

print("\nLettered Article identifiers:")
print(lettered_counts.to_string())

LETTERED ARTICLE IDENTIFIERS FOUND IN DATASET
Total lettered references : 2,702
Unique identifiers        : 244

Lettered Article identifiers:
31A     222
39B     156
31C     153
329B    150
300A    133
304A    112
304B     92
39D      82
323A     75
243Q     69
224A     68
51A      54
371D     48
243T     37
323B     36
26C      36
134A     34
243C     34
139A     32
2B       31
39A      30
243N     27
31B      25
48A      25
243R     24
21A      23
26A      22
239A     21
286L     20
236A     20
243F     20
19I      19
243G     19
173A     19
243D     19
191G     18
243S     18
19L      17
243B     15
119B     14
61A      13
349A     12
243U     12
39F      12
26B      12
246A     11
43A      11
133A     10
236B     10
243P     10
279A     10
65B      10
243O     10
350A      9
243A      9
243K      9
45F       9
26D       9
81D       8
14I       8
2A        8
20A       7
371F      7
243W      7
305A      7
233A      7
39E       7
342A      6
60A       6
269A      6
81B       6
40B  

In [ ]:
# ============================================================
# STEP 23: Inspect Suspicious Lettered Article Candidates
# ============================================================

suspicious_lettered = [
    "39B",
    "31C",
    "329B",
    "304A",
    "39D",
    "26C",
    "19I",
    "19L"
]

print("=" * 80)
print("SUSPICIOUS LETTERED ARTICLE CANDIDATES")
print("=" * 80)

for value in suspicious_lettered:

    print(f"\n{'-' * 80}")
    print(f"Candidate: {value}")
    print(f"{'-' * 80}")

    # Allow spaces between number and letter because of PDF extraction
    number_part = re.match(r"\d+", value).group()
    letter_part = re.search(r"[A-Z]+$", value).group()

    pattern = re.compile(
        rf"\barticles?\s*[-:]?\s*"
        rf"{number_part}\s*{letter_part}\b",
        re.IGNORECASE
    )

    examples_found = 0

    for text in model_df["judgement"]:

        match = pattern.search(text)

        if match:
            start = max(0, match.start() - 180)
            end = min(len(text), match.end() + 250)

            print(
                text[start:end]
                .replace("\n", " ")
            )

            examples_found += 1

            # Only inspect first 3 examples per candidate
            if examples_found >= 3:
                break

    if examples_found == 0:
        print("No examples found.")

SUSPICIOUS LETTERED ARTICLE CANDIDATES

--------------------------------------------------------------------------------
Candidate: 39B
--------------------------------------------------------------------------------
es. on the issue of equitable distribution of resources, in view of the fact that tisco held 55 of he total chromite resource of the country, the committee was of the opinion that article 39 b  and article 1.4 of the constitution must be taken into consideration for deciding the issue under section 8 3 of the act. thereafter the committee examined the requirements of section 8 3  of the act. the provision as it originally stood was amende
rs.  1995  6 scc 309 this court was to consider whether the alienation of government lands allotted to the scheduled castes was in violation of the constitutional objectives under articles 39 b  and 46. it was held that economic empowerment to the dalits. tribes and the poor as a part of distributive justice is a fundamental right assignm

In [ ]:
# ============================================================
# STEP 24: Canonical Inserted Constitutional Articles
# ============================================================

canonical_inserted_articles = {
    "2A",

    "21A",

    "31A",
    "31B",
    "31C",

    "39A",

    "43A",
    "48A",
    "51A",

    "239A",
    "239AA",

    "243A",
    "243B",
    "243C",
    "243D",
    "243E",
    "243F",
    "243G",
    "243H",
    "243I",
    "243J",
    "243K",
    "243L",
    "243M",
    "243N",
    "243O",
    "243P",
    "243Q",
    "243R",
    "243S",
    "243T",
    "243U",
    "243V",
    "243W",
    "243X",
    "243Y",
    "243Z",
    "243ZA",
    "243ZB",
    "243ZC",
    "243ZD",
    "243ZE",
    "243ZF",
    "243ZG",

    "244A",

    "300A",

    "323A",
    "323B",

    "330A",
    "332A",
    "334A",
    "342A",

    "371A",
    "371B",
    "371C",
    "371D",
    "371E",
    "371F",
    "371G",
    "371H",
    "371I",
    "371J",

    "372A",

    "378A",

    "394A"
}

# Create the complete validation set
canonical_articles = {
    str(i) for i in range(1, 396)
}

canonical_articles.update(canonical_inserted_articles)

print("=" * 80)
print("CANONICAL ARTICLE REFERENCE SET")
print("=" * 80)

print(
    "Total canonical identifiers :",
    len(canonical_articles)
)

print(
    "Inserted Article identifiers:",
    len(canonical_inserted_articles)
)

print("\nInserted Articles:")
print(
    sorted(
        canonical_inserted_articles,
        key=lambda x: (int(re.match(r"\d+", x).group()), x)
    )
)

CANONICAL ARTICLE REFERENCE SET
Total canonical identifiers : 460
Inserted Article identifiers: 65

Inserted Articles:
['2A', '21A', '31A', '31B', '31C', '39A', '43A', '48A', '51A', '239A', '239AA', '243A', '243B', '243C', '243D', '243E', '243F', '243G', '243H', '243I', '243J', '243K', '243L', '243M', '243N', '243O', '243P', '243Q', '243R', '243S', '243T', '243U', '243V', '243W', '243X', '243Y', '243Z', '243ZA', '243ZB', '243ZC', '243ZD', '243ZE', '243ZF', '243ZG', '244A', '300A', '323A', '323B', '330A', '332A', '334A', '342A', '371A', '371B', '371C', '371D', '371E', '371F', '371G', '371H', '371I', '371J', '372A', '378A', '394A']


In [ ]:
# ============================================================
# STEP 25: Test Canonical Article Recognition
# ============================================================

test_identifiers = [
    "39B",
    "39D",
    "31C",
    "329B",
    "304A",
    "26C",
    "19I",
    "19L",
    "300A",
    "134A",
    "134B",
    "323A",
    "323B",
    "21A",
    "239AA"
]

print("=" * 80)
print("CANONICAL ARTICLE IDENTIFIER TEST")
print("=" * 80)

for identifier in test_identifiers:

    if identifier in canonical_articles:
        status = "VALID CANONICAL ARTICLE"
    else:
        status = "NOT A CANONICAL ARTICLE"

    print(f"{identifier:8} -> {status}")

CANONICAL ARTICLE IDENTIFIER TEST
39B      -> NOT A CANONICAL ARTICLE
39D      -> NOT A CANONICAL ARTICLE
31C      -> VALID CANONICAL ARTICLE
329B     -> NOT A CANONICAL ARTICLE
304A     -> NOT A CANONICAL ARTICLE
26C      -> NOT A CANONICAL ARTICLE
19I      -> NOT A CANONICAL ARTICLE
19L      -> NOT A CANONICAL ARTICLE
300A     -> VALID CANONICAL ARTICLE
134A     -> NOT A CANONICAL ARTICLE
134B     -> NOT A CANONICAL ARTICLE
323A     -> VALID CANONICAL ARTICLE
323B     -> VALID CANONICAL ARTICLE
21A      -> VALID CANONICAL ARTICLE
239AA    -> VALID CANONICAL ARTICLE


In [ ]:
# ============================================================
# STEP 26: Improved Article Extraction Function - Version 2
# ============================================================

article_reference_pattern_v2 = re.compile(
    r"""
    \b
    (?:articles?|arts?)
    \.?
    \s*
    [-:]?
    \s*
    (
        \d{1,3}[a-zA-Z]?
        (?:
            \s*
            (?:,|and)
            \s*
            \d{1,3}[a-zA-Z]?
        )*
    )
    """,
    re.IGNORECASE | re.VERBOSE
)


def normalize_article_candidate(number, letter=None):
    """
    Normalize an Article candidate.

    Rules:
    - Base number is always retained.
    - A letter is attached only when the resulting
      identifier is a canonical inserted Article.
    """

    number = str(number).strip()

    if letter:
        combined = f"{number}{letter}".upper()

        # Keep letter only if it is a genuine
        # canonical inserted Article.
        if combined in canonical_articles:
            return combined

    return number


def extract_articles_v2(text):
    """
    Extract Indian Constitutional Article references.

    Handles:
    - Article 226
    - Articles 14 and 19
    - Article 300A
    - Article 300 a
    - Article 16 4a
    - Article 19 1 g
    - Article-136
    - Art. 133

    Returns:
        Comma-separated unique Article identifiers.
    """

    if pd.isna(text) or not str(text).strip():
        return ""

    text = str(text)

    extracted = []

    for match in article_reference_pattern_v2.finditer(text):

        expression = match.group(1)

        # Find individual number/letter tokens
        tokens = re.findall(
            r"\d{1,3}[a-zA-Z]?",
            expression
        )

        for token in tokens:

            token = token.upper()

            # Case 1:
            # Directly attached form such as 300A
            direct_match = re.fullmatch(
                r"(\d{1,3})([A-Z])",
                token
            )

            if direct_match:

                number = direct_match.group(1)
                letter = direct_match.group(2)

                normalized = normalize_article_candidate(
                    number,
                    letter
                )

            else:
                # Normal numeric Article
                normalized = token

            # Validate base/combined identifier
            if normalized in canonical_articles:
                extracted.append(normalized)

    # Remove duplicates while preserving order
    unique_articles = list(dict.fromkeys(extracted))

    return ",".join(unique_articles)


print("=" * 80)
print("ARTICLE EXTRACTION FUNCTION V2 CREATED")
print("=" * 80)

print("Function: extract_articles_v2()")
print("Canonical identifiers:", len(canonical_articles))

ARTICLE EXTRACTION FUNCTION V2 CREATED
Function: extract_articles_v2()
Canonical identifiers: 460


In [ ]:
# ============================================================
# STEP 27: Test Article Extraction Function V2
# ============================================================

test_cases_v2 = [
    "Article 226 of the Constitution",
    "Article 136 of the Constitution",
    "Articles 14 and 19 of the Constitution",
    "Articles 14, 15 and 16 of the Constitution",

    # Clause references
    "Article 39 b of the Constitution",
    "Article 39 d of the Constitution",
    "Article 304 a of the Constitution",
    "Article 329 b of the Constitution",
    "Article 26 c and d of the Constitution",
    "Article 19 i g of the Constitution",

    # Genuine inserted Articles
    "Article 31C of the Constitution",
    "Article 300A of the Constitution",
    "Article 323A and Article 323B of the Constitution",
    "Article 21A of the Constitution",
    "Article 239AA of the Constitution",

    # PDF-style separated inserted Article
    "Article 300 a of the Constitution",
    "Article 31 c of the Constitution",

    # Multiple references
    "Articles 226 and 227 of the Constitution",
    "Articles 14, 15 and 16 of the Constitution",
    "Articles 134, 134a, 134b and 142",

    # Duplicate
    "Article 226 and 226",

    # Other formats
    "Article-136 of the Constitution",
    "Art. 133 of the Constitution",

    # False positives
    "articles into possession and sealed them",
    "articles of material described in sub-item",
    "articles stolen from the house",
    "Article shall prevent the State from making any law",
]

print("=" * 80)
print("ARTICLE EXTRACTION V2 TEST")
print("=" * 80)

for i, text in enumerate(test_cases_v2, start=1):

    result = extract_articles_v2(text)

    print(f"\n{i:02}. Input  : {text}")
    print(f"    Output : {result}")

ARTICLE EXTRACTION V2 TEST

01. Input  : Article 226 of the Constitution
    Output : 226

02. Input  : Article 136 of the Constitution
    Output : 136

03. Input  : Articles 14 and 19 of the Constitution
    Output : 14,19

04. Input  : Articles 14, 15 and 16 of the Constitution
    Output : 14,15,16

05. Input  : Article 39 b of the Constitution
    Output : 39

06. Input  : Article 39 d of the Constitution
    Output : 39

07. Input  : Article 304 a of the Constitution
    Output : 304

08. Input  : Article 329 b of the Constitution
    Output : 329

09. Input  : Article 26 c and d of the Constitution
    Output : 26

10. Input  : Article 19 i g of the Constitution
    Output : 19

11. Input  : Article 31C of the Constitution
    Output : 31C

12. Input  : Article 300A of the Constitution
    Output : 300A

13. Input  : Article 323A and Article 323B of the Constitution
    Output : 323A,323B

14. Input  : Article 21A of the Constitution
    Output : 21A

15. Input  : Article 239AA 

In [ ]:
# ============================================================
# STEP 28: Investigate Ambiguous Lettered Article References
# ============================================================

patterns_to_check = {
    "31C": r"\barticles?\s*31\s*c\b",
    "134A": r"\barticles?\s*134\s*a\b",
    "134B": r"\barticles?\s*134\s*b\b",
    "239AA": r"\barticles?\s*239\s*a{2}\b",
    "300A": r"\barticles?\s*300\s*a\b"
}

print("=" * 80)
print("AMBIGUOUS ARTICLE REFERENCE INVESTIGATION")
print("=" * 80)

for label, pattern_text in patterns_to_check.items():

    print(f"\n{'-' * 80}")
    print(f"Pattern: {label}")
    print(f"{'-' * 80}")

    pattern = re.compile(
        pattern_text,
        re.IGNORECASE
    )

    count = 0

    for text in model_df["judgement"]:

        match = pattern.search(text)

        if match:

            start = max(0, match.start() - 200)
            end = min(len(text), match.end() + 300)

            print(
                text[start:end]
                .replace("\n", " ")
            )

            count += 1

            if count >= 5:
                break

    print(f"\nExamples displayed: {count}")

AMBIGUOUS ARTICLE REFERENCE INVESTIGATION

--------------------------------------------------------------------------------
Pattern: 31C
--------------------------------------------------------------------------------
 illusory and violating article 31 2  of the constitution as it stands after the constitution  twentyfifth  amendment  act, 1971? 2. whether its constitutionality is not immune from attack in view of article 31c because in reality it does not give effect to the policy of the state towards securing the principles specified in clause  b  of article 39 despite the declaration to that effect contained in section 30 of the act? 10. applying the ratio of the majority judgment in kesavananda bharati v. state of k
which were dis- missed, relying on its earlier full bench decision in nitya- nanda guru v. state of orissa,  a.i.r. 1983 orissa 54 . before this court it was contended that  1  the protec- tion under article 31 c  would not be available to section 37 b  of the act and i

In [ ]:
# ============================================================
# STEP 29: Update Canonical Article Reference Set
# ============================================================

# Article 134A is a genuine Constitutional Article.
# It was missing from our initial canonical set.

canonical_inserted_articles.add("134A")

# Rebuild complete canonical set
canonical_articles = {
    str(i) for i in range(1, 396)
}

canonical_articles.update(canonical_inserted_articles)

print("=" * 80)
print("UPDATED CANONICAL ARTICLE REFERENCE SET")
print("=" * 80)

print(
    "Total canonical identifiers :",
    len(canonical_articles)
)

print(
    "Inserted Article identifiers:",
    len(canonical_inserted_articles)
)

print("\nNewly verified Article:")
print("134A ->", "134A" in canonical_articles)

print("\n134B ->", "134B" in canonical_articles)

UPDATED CANONICAL ARTICLE REFERENCE SET
Total canonical identifiers : 461
Inserted Article identifiers: 66

Newly verified Article:
134A -> True

134B -> False


In [ ]:
# ============================================================
# STEP 30: Article Extraction Function - Version 3
# ============================================================

# Match:
# Article / Articles / Art / Arts
# followed by a number
# optionally followed by letters
# optionally separated by spaces
article_reference_pattern_v3 = re.compile(
    r"""
    \b
    (?:articles?|arts?)
    \.?
    \s*
    [-:]?
    \s*
    (
        \d{1,3}
        (?:\s*[a-zA-Z]{1,2})?
    )
    """,
    re.IGNORECASE | re.VERBOSE
)


def normalize_article_identifier_v3(number, letters=None):
    """
    Normalize Article identifier.

    Examples:
        300 + A  -> 300A
        239 + AA -> 239AA
        39 + B   -> 39
        19 + I   -> 19
    """

    number = str(number).strip()

    if letters:
        letters = re.sub(r"\s+", "", letters).upper()

        combined = f"{number}{letters}"

        # Keep the letter only if it is a canonical
        # Constitutional Article.
        if combined in canonical_articles:
            return combined

    return number


def extract_articles_v3(text):
    """
    Extract Indian Constitutional Article references.

    Handles:
    - Article 226
    - Articles 14 and 19
    - Article 31C
    - Article 300A
    - Article 300 a
    - Article 239AA
    - Article 239 aa
    - Article 39 b
    - Article 19 i g

    Returns:
        Comma-separated unique Article identifiers.
    """

    if pd.isna(text) or not str(text).strip():
        return ""

    text = str(text)

    extracted_articles = []

    # --------------------------------------------------------
    # Find Article references
    # --------------------------------------------------------

    for match in article_reference_pattern_v3.finditer(text):

        expression = match.group(1)

        # Separate number and optional letters
        identifier_match = re.fullmatch(
            r"(\d{1,3})\s*([a-zA-Z]{1,2})?",
            expression
        )

        if not identifier_match:
            continue

        number = identifier_match.group(1)
        letters = identifier_match.group(2)

        normalized = normalize_article_identifier_v3(
            number,
            letters
        )

        # Validate numeric base
        if normalized in canonical_articles:
            extracted_articles.append(normalized)

    # --------------------------------------------------------
    # Handle multiple Article references
    # --------------------------------------------------------

    # The first pattern captures only the first Article.
    # Now inspect the short continuation after it for:
    #
    #   and 19
    #   , 15 and 16
    #
    # This allows extraction of multiple Articles.

    multi_pattern = re.compile(
        r"""
        \b
        (?:articles?|arts?)
        \.?
        \s*
        [-:]?
        \s*
        \d{1,3}(?:\s*[a-zA-Z]{1,2})?
        (
            (?:
                \s*(?:,|and)\s*
                \d{1,3}(?:\s*[a-zA-Z]{1,2})?
            )+
        )
        """,
        re.IGNORECASE | re.VERBOSE
    )

    for match in multi_pattern.finditer(text):

        continuation = match.group(1)

        continuation_values = re.findall(
            r"\d{1,3}(?:\s*[a-zA-Z]{1,2})?",
            continuation
        )

        for value in continuation_values:

            value_match = re.fullmatch(
                r"(\d{1,3})\s*([a-zA-Z]{1,2})?",
                value
            )

            if not value_match:
                continue

            number = value_match.group(1)
            letters = value_match.group(2)

            normalized = normalize_article_identifier_v3(
                number,
                letters
            )

            if normalized in canonical_articles:
                extracted_articles.append(normalized)

    # --------------------------------------------------------
    # Remove duplicates while preserving order
    # --------------------------------------------------------

    unique_articles = list(
        dict.fromkeys(extracted_articles)
    )

    return ",".join(unique_articles)


print("=" * 80)
print("ARTICLE EXTRACTION FUNCTION V3 CREATED")
print("=" * 80)

print("Function: extract_articles_v3()")
print("Canonical identifiers:", len(canonical_articles))

ARTICLE EXTRACTION FUNCTION V3 CREATED
Function: extract_articles_v3()
Canonical identifiers: 461


In [ ]:
# ============================================================
# STEP 31: Test Article Extraction Function V3
# ============================================================

test_cases_v3 = [
    # Basic Articles
    "Article 226 of the Constitution",
    "Article 136 of the Constitution",
    "Article 14 of the Constitution",

    # Multiple Articles
    "Articles 14 and 19 of the Constitution",
    "Articles 14, 15 and 16 of the Constitution",
    "Articles 226 and 227 of the Constitution",

    # Clause references - should keep BASE Article
    "Article 39 b of the Constitution",
    "Article 39 d of the Constitution",
    "Article 304 a of the Constitution",
    "Article 329 b of the Constitution",
    "Article 26 c and d of the Constitution",
    "Article 19 i g of the Constitution",

    # Genuine inserted Articles
    "Article 31C of the Constitution",
    "Article 300A of the Constitution",
    "Article 323A of the Constitution",
    "Article 323B of the Constitution",
    "Article 21A of the Constitution",
    "Article 239AA of the Constitution",

    # PDF-separated inserted Articles
    "Article 300 a of the Constitution",
    "Article 239 aa of the Constitution",
    "Article 134 a of the Constitution",
    "Article 31 c of the Constitution",

    # Multiple inserted Articles
    "Articles 134, 134a, 134b and 142",

    # Duplicate
    "Article 226 and 226",

    # Other formats
    "Article-136 of the Constitution",
    "Art. 133 of the Constitution",

    # False positives
    "articles into possession and sealed them",
    "articles of material described in sub-item",
    "articles stolen from the house",
    "Article shall prevent the State from making any law",

    # Other legal instrument
    "Article 134a of the Limitation Act",
    "Article 134b of the Limitation Act",
]

print("=" * 80)
print("ARTICLE EXTRACTION V3 TEST")
print("=" * 80)

for i, text in enumerate(test_cases_v3, start=1):

    result = extract_articles_v3(text)

    print(f"\n{i:02}. Input  : {text}")
    print(f"    Output : {result}")

ARTICLE EXTRACTION V3 TEST

01. Input  : Article 226 of the Constitution
    Output : 226

02. Input  : Article 136 of the Constitution
    Output : 136

03. Input  : Article 14 of the Constitution
    Output : 14

04. Input  : Articles 14 and 19 of the Constitution
    Output : 14,19

05. Input  : Articles 14, 15 and 16 of the Constitution
    Output : 14,15

06. Input  : Articles 226 and 227 of the Constitution
    Output : 226,227

07. Input  : Article 39 b of the Constitution
    Output : 39

08. Input  : Article 39 d of the Constitution
    Output : 39

09. Input  : Article 304 a of the Constitution
    Output : 304

10. Input  : Article 329 b of the Constitution
    Output : 329

11. Input  : Article 26 c and d of the Constitution
    Output : 26

12. Input  : Article 19 i g of the Constitution
    Output : 19

13. Input  : Article 31C of the Constitution
    Output : 31C

14. Input  : Article 300A of the Constitution
    Output : 300A

15. Input  : Article 323A of the Constituti

In [ ]:
# ============================================================
# STEP 32: Article Extraction Function - Version 3.1
# Fix Multi-Article Extraction
# ============================================================

article_reference_pattern_v31 = re.compile(
    r"""
    \b
    (?:articles?|arts?)
    \.?
    \s*
    [-:]?
    \s*
    (
        \d{1,3}
        (?:\s*[a-zA-Z]{1,2})?
        (?:
            \s*(?:,|and)\s*
            \d{1,3}
            (?:\s*[a-zA-Z]{1,2})?
        )*
    )
    """,
    re.IGNORECASE | re.VERBOSE
)


def normalize_article_identifier_v31(number, letters=None):
    """
    Normalize an Article identifier.

    If number + letters is a canonical inserted Article,
    preserve the letters.

    Otherwise keep only the base Article number.
    """

    number = str(number).strip()

    if letters:
        letters = re.sub(r"\s+", "", letters).upper()

        combined = f"{number}{letters}"

        if combined in canonical_articles:
            return combined

    return number


def extract_articles_v31(text):
    """
    Extract Indian Constitutional Article references.

    Handles:
    - Single Articles
    - Multiple Articles
    - Lettered Articles
    - PDF-separated Article identifiers
    - Clause references
    - Duplicate references

    Returns:
        Comma-separated unique Article identifiers.
    """

    if pd.isna(text) or not str(text).strip():
        return ""

    text = str(text)

    extracted_articles = []

    # --------------------------------------------------------
    # Find Article expressions
    # --------------------------------------------------------

    for match in article_reference_pattern_v31.finditer(text):

        expression = match.group(1)

        # Extract each individual candidate
        candidates = re.findall(
            r"\d{1,3}(?:\s*[a-zA-Z]{1,2})?",
            expression
        )

        for candidate in candidates:

            identifier_match = re.fullmatch(
                r"(\d{1,3})\s*([a-zA-Z]{1,2})?",
                candidate
            )

            if not identifier_match:
                continue

            number = identifier_match.group(1)
            letters = identifier_match.group(2)

            normalized = normalize_article_identifier_v31(
                number,
                letters
            )

            if normalized in canonical_articles:
                extracted_articles.append(normalized)

    # --------------------------------------------------------
    # Remove duplicates while preserving order
    # --------------------------------------------------------

    unique_articles = list(
        dict.fromkeys(extracted_articles)
    )

    return ",".join(unique_articles)


print("=" * 80)
print("ARTICLE EXTRACTION FUNCTION V3.1 CREATED")
print("=" * 80)

print("Function: extract_articles_v31()")
print("Canonical identifiers:", len(canonical_articles))

ARTICLE EXTRACTION FUNCTION V3.1 CREATED
Function: extract_articles_v31()
Canonical identifiers: 461


In [ ]:
# ============================================================
# STEP 33: Test Multiple Article Extraction - V3.1
# ============================================================

multi_article_tests = [
    "Articles 14 and 19 of the Constitution",
    "Articles 14, 15 and 16 of the Constitution",
    "Articles 14, 15, 16, 21, 38, 39 and 46 of the Constitution",
    "Articles 226 and 227 of the Constitution",
    "Articles 249, 250, 252 and 357 of the Constitution",
    "Articles 132, 133, 134 and 134a of the Constitution",
    "Articles 134, 134a, 134b and 142",
    "Articles 323a and 323b of the Constitution",
    "Articles 141 and 142 of the Constitution",
    "Articles 14, 19 and 21 of the Constitution",
    "Article 226 and 226 of the Constitution",
    "Articles 14 and 14 of the Constitution",
]

print("=" * 80)
print("MULTIPLE ARTICLE EXTRACTION TEST - V3.1")
print("=" * 80)

for i, text in enumerate(multi_article_tests, start=1):

    result = extract_articles_v31(text)

    print(f"\n{i:02}. Input  : {text}")
    print(f"    Output : {result}")

MULTIPLE ARTICLE EXTRACTION TEST - V3.1

01. Input  : Articles 14 and 19 of the Constitution
    Output : 14

02. Input  : Articles 14, 15 and 16 of the Constitution
    Output : 14,15

03. Input  : Articles 14, 15, 16, 21, 38, 39 and 46 of the Constitution
    Output : 14,15,16,21,38,39

04. Input  : Articles 226 and 227 of the Constitution
    Output : 226

05. Input  : Articles 249, 250, 252 and 357 of the Constitution
    Output : 249,250,252

06. Input  : Articles 132, 133, 134 and 134a of the Constitution
    Output : 132,133,134

07. Input  : Articles 134, 134a, 134b and 142
    Output : 134,134A,142

08. Input  : Articles 323a and 323b of the Constitution
    Output : 323A,323B

09. Input  : Articles 141 and 142 of the Constitution
    Output : 141

10. Input  : Articles 14, 19 and 21 of the Constitution
    Output : 14,19

11. Input  : Article 226 and 226 of the Constitution
    Output : 226

12. Input  : Articles 14 and 14 of the Constitution
    Output : 14


In [ ]:
# ============================================================
# STEP 34: Article Extraction Function - Version 4
# Simplified Multi-Article Parsing
# ============================================================

# Detect the beginning of an Article reference
article_start_pattern_v4 = re.compile(
    r"\b(?:articles?|arts?)\.?\s*[-:]?\s*",
    re.IGNORECASE
)


def normalize_article_identifier_v4(number, letters=None):
    """
    Normalize an Article identifier.

    Examples:
        300 + A  -> 300A
        239 + AA -> 239AA
        39 + B   -> 39
        19 + I   -> 19
    """

    number = str(number).strip()

    if letters:
        letters = re.sub(r"\s+", "", letters).upper()

        combined = f"{number}{letters}"

        if combined in canonical_articles:
            return combined

    return number


def extract_articles_v4(text):
    """
    Extract Constitutional Article references.

    Uses a two-stage approach:
    1. Find Article/Articles markers.
    2. Inspect the short sequence immediately following
       the marker for Article numbers.

    Returns:
        Comma-separated unique Article identifiers.
    """

    if pd.isna(text) or not str(text).strip():
        return ""

    text = str(text)

    extracted_articles = []

    # --------------------------------------------------------
    # Find every Article / Articles occurrence
    # --------------------------------------------------------

    for match in article_start_pattern_v4.finditer(text):

        start = match.end()

        # Look only at a short window after "Article(s)"
        # to avoid consuming the rest of the judgment.
        context = text[start:start + 180]

        # Stop at obvious sentence boundaries
        context = re.split(
            r"[.;:!?]",
            context
        )[0]

        # ----------------------------------------------------
        # Find Article candidates
        # ----------------------------------------------------

        candidates = re.findall(
            r"\d{1,3}(?:\s*[a-zA-Z]{1,2})?",
            context
        )

        for candidate in candidates:

            candidate = candidate.strip()

            identifier_match = re.fullmatch(
                r"(\d{1,3})\s*([a-zA-Z]{1,2})?",
                candidate
            )

            if not identifier_match:
                continue

            number = identifier_match.group(1)
            letters = identifier_match.group(2)

            normalized = normalize_article_identifier_v4(
                number,
                letters
            )

            if normalized in canonical_articles:
                extracted_articles.append(normalized)

    # --------------------------------------------------------
    # Remove duplicates while preserving order
    # --------------------------------------------------------

    unique_articles = list(
        dict.fromkeys(extracted_articles)
    )

    return ",".join(unique_articles)


print("=" * 80)
print("ARTICLE EXTRACTION FUNCTION V4 CREATED")
print("=" * 80)

print("Function: extract_articles_v4()")
print("Canonical identifiers:", len(canonical_articles))

ARTICLE EXTRACTION FUNCTION V4 CREATED
Function: extract_articles_v4()
Canonical identifiers: 461


In [ ]:
# ============================================================
# STEP 35: Test Multiple Article Extraction - V4
# ============================================================

multi_article_tests_v4 = [
    "Articles 14 and 19 of the Constitution",
    "Articles 14, 15 and 16 of the Constitution",
    "Articles 14, 15, 16, 21, 38, 39 and 46 of the Constitution",
    "Articles 226 and 227 of the Constitution",
    "Articles 249, 250, 252 and 357 of the Constitution",
    "Articles 132, 133, 134 and 134a of the Constitution",
    "Articles 134, 134a, 134b and 142",
    "Articles 323a and 323b of the Constitution",
    "Articles 141 and 142 of the Constitution",
    "Articles 14, 19 and 21 of the Constitution",
    "Article 226 and 226 of the Constitution",
    "Articles 14 and 14 of the Constitution",
]

print("=" * 80)
print("MULTIPLE ARTICLE EXTRACTION TEST - V4")
print("=" * 80)

for i, text in enumerate(multi_article_tests_v4, start=1):

    result = extract_articles_v4(text)

    print(f"\n{i:02}. Input  : {text}")
    print(f"    Output : {result}")

MULTIPLE ARTICLE EXTRACTION TEST - V4

01. Input  : Articles 14 and 19 of the Constitution
    Output : 14,19

02. Input  : Articles 14, 15 and 16 of the Constitution
    Output : 14,15,16

03. Input  : Articles 14, 15, 16, 21, 38, 39 and 46 of the Constitution
    Output : 14,15,16,21,38,39,46

04. Input  : Articles 226 and 227 of the Constitution
    Output : 226,227

05. Input  : Articles 249, 250, 252 and 357 of the Constitution
    Output : 249,250,252,357

06. Input  : Articles 132, 133, 134 and 134a of the Constitution
    Output : 132,133,134,134A

07. Input  : Articles 134, 134a, 134b and 142
    Output : 134,134A,142

08. Input  : Articles 323a and 323b of the Constitution
    Output : 323A,323B

09. Input  : Articles 141 and 142 of the Constitution
    Output : 141,142

10. Input  : Articles 14, 19 and 21 of the Constitution
    Output : 14,19,21

11. Input  : Article 226 and 226 of the Constitution
    Output : 226

12. Input  : Articles 14 and 14 of the Constitution
    Ou

In [ ]:
# ============================================================
# STEP 36: Analyze Constitutional Context of Article References
# ============================================================

print("=" * 80)
print("CONSTITUTIONAL ARTICLE CONTEXT ANALYSIS")
print("=" * 80)

article_context_pattern = re.compile(
    r"\b(?:articles?|arts?)\.?\s*[-:]?\s*\d{1,3}",
    re.IGNORECASE
)

constitution_context_count = 0
total_article_contexts = 0

for text in model_df["judgement"]:

    if pd.isna(text):
        continue

    text = str(text)

    for match in article_context_pattern.finditer(text):

        total_article_contexts += 1

        start = max(0, match.start() - 100)
        end = min(len(text), match.end() + 150)

        context = text[start:end].lower()

        if "constitution" in context:
            constitution_context_count += 1

print("Total Article references inspected :", total_article_contexts)
print("References near 'constitution'      :", constitution_context_count)

if total_article_contexts > 0:
    percentage = (
        constitution_context_count /
        total_article_contexts
    ) * 100

    print(
        f"Percentage with constitutional context: "
        f"{percentage:.2f}%"
    )

CONSTITUTIONAL ARTICLE CONTEXT ANALYSIS
Total Article references inspected : 77386
References near 'constitution'      : 44886
Percentage with constitutional context: 58.00%


In [ ]:
# ============================================================
# STEP 37: Diagnostic Article Extraction on Full Dataset
# ============================================================

print("=" * 80)
print("RUNNING ARTICLE EXTRACTION DIAGNOSTIC")
print("=" * 80)

print(f"Total judgments: {len(model_df):,}")
print("\nExtracting Article references...")

article_diagnostic = model_df["judgement"].apply(
    extract_articles_v4
)

print("\nExtraction completed.")

# ------------------------------------------------------------
# Basic statistics
# ------------------------------------------------------------

non_empty = (article_diagnostic != "").sum()
empty = (article_diagnostic == "").sum()

print("\n" + "=" * 80)
print("ARTICLE EXTRACTION DIAGNOSTIC RESULTS")
print("=" * 80)

print(f"Total Judgments              : {len(model_df):,}")
print(f"Judgments with Article(s)    : {non_empty:,}")
print(f"Judgments without Article(s) : {empty:,}")

if len(model_df) > 0:

    percentage = (
        non_empty / len(model_df)
    ) * 100

    print(
        f"Percentage with Article(s)  : "
        f"{percentage:.2f}%"
    )

# ------------------------------------------------------------
# Distribution of number of Articles per judgment
# ------------------------------------------------------------

article_count = article_diagnostic.apply(
    lambda x: len(x.split(",")) if x else 0
)

print("\n" + "=" * 80)
print("NUMBER OF ARTICLES PER JUDGMENT")
print("=" * 80)

print(
    article_count.value_counts()
    .sort_index()
    .head(20)
)

# ------------------------------------------------------------
# Most common extracted Article identifiers
# ------------------------------------------------------------

all_articles = []

for value in article_diagnostic:

    if value:
        all_articles.extend(
            value.split(",")
        )

article_frequency = pd.Series(
    all_articles
).value_counts()

print("\n" + "=" * 80)
print("TOP 30 EXTRACTED ARTICLE IDENTIFIERS")
print("=" * 80)

print(article_frequency.head(30))

RUNNING ARTICLE EXTRACTION DIAGNOSTIC
Total judgments: 42,169

Extracting Article references...

Extraction completed.

ARTICLE EXTRACTION DIAGNOSTIC RESULTS
Total Judgments              : 42,169
Judgments with Article(s)    : 14,685
Judgments without Article(s) : 27,484
Percentage with Article(s)  : 34.82%

NUMBER OF ARTICLES PER JUDGMENT
judgement
0     27484
1      4864
2      2788
3      1729
4      1318
5       931
6       750
7       514
8       421
9       311
10      209
11      196
12      145
13      100
14       94
15       55
16       58
17       39
18       36
19       22
Name: count, dtype: int64

TOP 30 EXTRACTED ARTICLE IDENTIFIERS
1      4139
226    4085
14     2960
136    2687
2      2420
3      1935
19     1305
4      1162
5      1160
16     1157
227    1131
6      1091
32     1082
195    1014
196     939
21      936
8       839
133     829
7       753
9       700
309     658
12      607
142     594
311     583
197     566
15      477
31      467
20      464
198     

In [ ]:
# ============================================================
# STEP 38: Audit Extracted Article References
# ============================================================

print("=" * 80)
print("ARTICLE EXTRACTION QUALITY AUDIT")
print("=" * 80)

# Select judgments where Articles were extracted
article_indices = article_diagnostic[
    article_diagnostic != ""
].index

print(
    f"Judgments available for audit: "
    f"{len(article_indices):,}"
)

# Inspect first 15 examples
for i, idx in enumerate(article_indices[:15], start=1):

    articles_found = article_diagnostic.loc[idx]
    judgment = model_df.loc[idx, "judgement"]

    print("\n" + "-" * 80)
    print(f"Example {i}")
    print(f"Extracted Article(s): {articles_found}")
    print("-" * 80)

    # Show the first 700 characters containing
    # the first Article occurrence
    match = re.search(
        r"\b(?:articles?|arts?)\.?\s*[-:]?\s*\d{1,3}",
        judgment,
        re.IGNORECASE
    )

    if match:

        start = max(0, match.start() - 150)
        end = min(len(judgment), match.end() + 550)

        print(
            judgment[start:end]
            .replace("\n", " ")
        )

    else:
        print(judgment[:700])

ARTICLE EXTRACTION QUALITY AUDIT
Judgments available for audit: 14,685

--------------------------------------------------------------------------------
Example 1
Extracted Article(s): 226,199
--------------------------------------------------------------------------------
atter for hearing and refused to pass an order on the maintainability of the show cause notice alone, the legal heirs approached the high court under article 226 of the constitution by filing a writ petition in january, 1990. the learned single judge of the high court quashed the proceedings against the legal heirs stating that the central excises and salt act did not contain any provisions for continuing assessment proceedings against a dead person. against this, revenue went in appeal. the division bench of the high court of kerala reversed the single judges judgment. 5. shri rajshekhar rao, learned counsel appearing for the legal heirs made submissions before us with great clarity and persuasiveness

-------------

In [ ]:
# ============================================================
# STEP 39: Inspect Current Canonical Inserted Article Set
# ============================================================

print("=" * 80)
print("CURRENT CANONICAL INSERTED ARTICLE SET")
print("=" * 80)

for article in sorted(
    canonical_inserted_articles,
    key=lambda x: (int(re.match(r"\d+", x).group()), x)
):
    print(article)

CURRENT CANONICAL INSERTED ARTICLE SET
2A
21A
31A
31B
31C
39A
43A
48A
51A
134A
239A
239AA
243A
243B
243C
243D
243E
243F
243G
243H
243I
243J
243K
243L
243M
243N
243O
243P
243Q
243R
243S
243T
243U
243V
243W
243X
243Y
243Z
243ZA
243ZB
243ZC
243ZD
243ZE
243ZF
243ZG
244A
300A
323A
323B
330A
332A
334A
342A
371A
371B
371C
371D
371E
371F
371G
371H
371I
371J
372A
378A
394A


In [ ]:
# ============================================================
# STEP 40: Analyze Real Article Reference Structures
# ============================================================

article_structure_pattern = re.compile(
    r"""
    \b
    (?:articles?|arts?)
    \.?
    \s*
    [-:]?
    \s*
    (\d{1,3})
    (
        (?:\s+[a-zA-Z]{1,2})?
        (?:\s+\d{1,2})?
        (?:\s+[a-zA-Z]{1,2})?
    )?
    """,
    re.IGNORECASE | re.VERBOSE
)

structures = []

for text in model_df["judgement"]:

    if pd.isna(text):
        continue

    text = str(text)

    for match in article_structure_pattern.finditer(text):

        base = match.group(1)
        continuation = (match.group(2) or "").strip()

        structure = f"{base} {continuation}".strip()

        structures.append(structure.lower())

structure_counts = pd.Series(
    structures
).value_counts()

print("=" * 80)
print("REAL ARTICLE REFERENCE STRUCTURE ANALYSIS")
print("=" * 80)

print(
    f"Total structures extracted: "
    f"{len(structures):,}"
)

print(
    f"Unique structures: "
    f"{structure_counts.nunique():,}"
)

print("\nTop 100 structures:")
print(structure_counts.head(100))

REAL ARTICLE REFERENCE STRUCTURE ANALYSIS
Total structures extracted: 76,998
Unique structures: 168

Top 100 structures:
226 of th    5471
14 of th     3156
136 of th    3135
14 an        2214
14           1714
             ... 
286            91
136 an         89
224            87
19 1           86
134 of th      86
Name: count, Length: 100, dtype: int64


In [ ]:
# ============================================================
# STEP 42: Create Final Case Article Feature
# ============================================================

print("=" * 80)
print("CREATING CASE ARTICLE FEATURE")
print("=" * 80)

# Create a copy so the original model_df remains untouched
model_df_final = model_df.copy()

print("Extracting Constitutional Article references...")

model_df_final["case_article"] = model_df_final["judgement"].apply(
    extract_articles_v4
)

print("\nArticle extraction completed.")

print("\n" + "=" * 80)
print("CASE ARTICLE FEATURE SUMMARY")
print("=" * 80)

print("Dataset Shape:", model_df_final.shape)

print("\nColumns:")
print(model_df_final.columns.tolist())

print("\nNon-empty case_article:")
print(
    (model_df_final["case_article"] != "").sum()
)

print("\nEmpty case_article:")
print(
    (model_df_final["case_article"] == "").sum()
)

print("\nSample values:")
display(
    model_df_final[
        ["case_article", "label"]
    ].head(20)
)

CREATING CASE ARTICLE FEATURE
Extracting Constitutional Article references...

Article extraction completed.

CASE ARTICLE FEATURE SUMMARY
Dataset Shape: (42169, 10)

Columns:
['case_category', 'case_type', 'case_info', 'judgement', 'label', 'char_count', 'word_count', 'sentence_count', 'cleaned_judgement', 'case_article']

Non-empty case_article:
14685

Empty case_article:
27484

Sample values:


,case_article,label
0,"226,199",1
1,"311,2",1
2,136,1
3,,1
4,,0
5,136,0
6,,0
7,,1
8,,1
9,,0


In [ ]:
# ============================================================
# STEP 43: Create Conservative Article Extraction Function
# ============================================================

def extract_articles_final(text):
    """
    Extract Constitutional Article references from legal text.

    Conservative approach:
    - Handles Article / Articles / Art.
    - Handles multiple Article references.
    - Handles inserted Articles such as 300A and 239AA.
    - Removes clause/sub-clause numbers.
    - Removes duplicates.
    - Uses the canonical Article reference set.
    """

    if pd.isna(text):
        return ""

    text = str(text)

    found_articles = []

    # --------------------------------------------------------
    # Normalize common PDF formatting
    # --------------------------------------------------------

    text = re.sub(r"\s+", " ", text)

    # --------------------------------------------------------
    # Pattern 1:
    # Explicit Article references
    #
    # Examples:
    # Article 226
    # Article-136
    # Art. 133
    # Articles 14 and 19
    # --------------------------------------------------------

    pattern = re.compile(
        r"\b(?:articles?|arts?)\.?\s*[-:]?\s*"
        r"(\d{1,3}(?:\s*[a-zA-Z]{1,2})?)",
        re.IGNORECASE
    )

    matches = pattern.finditer(text)

    for match in matches:

        raw = match.group(1).strip()

        # Remove internal spaces
        normalized = raw.replace(" ", "").upper()

        # ----------------------------------------------------
        # Direct canonical match
        # ----------------------------------------------------

        if normalized in canonical_articles:
            found_articles.append(normalized)
            continue

        # ----------------------------------------------------
        # Base number + letter
        # Example:
        # 300 a -> 300A
        # 239 aa -> 239AA
        # 31 c -> 31C
        # ----------------------------------------------------

        base_match = re.match(
            r"^(\d{1,3})([A-Z]{1,2})$",
            normalized
        )

        if base_match:

            candidate = (
                base_match.group(1)
                + base_match.group(2)
            )

            if candidate in canonical_articles:
                found_articles.append(candidate)

                continue

            # Otherwise use only the base number
            base = base_match.group(1)

            if base in canonical_articles:
                found_articles.append(base)

            continue

        # ----------------------------------------------------
        # Numeric Article
        # ----------------------------------------------------

        if normalized.isdigit():

            if normalized in canonical_articles:
                found_articles.append(normalized)

    # --------------------------------------------------------
    # Remove duplicates while preserving order
    # --------------------------------------------------------

    found_articles = list(
        dict.fromkeys(found_articles)
    )

    return ",".join(found_articles)


print("=" * 80)
print("FINAL ARTICLE EXTRACTION FUNCTION CREATED")
print("=" * 80)

print("Function: extract_articles_final()")
print(
    "Canonical identifiers:",
    len(canonical_articles)
)

FINAL ARTICLE EXTRACTION FUNCTION CREATED
Function: extract_articles_final()
Canonical identifiers: 461


In [ ]:
# ============================================================
# STEP 43B: Test Final Article Extractor
# ============================================================

final_article_tests = [
    "Article 226 of the Constitution",
    "Article 136 of the Constitution",
    "Articles 14 and 19 of the Constitution",
    "Articles 14, 15 and 16 of the Constitution",
    "Article 311 2 of the Constitution",
    "Article 133 1 a of the Constitution",
    "Article 19 1 g of the Constitution",
    "Article 16 4a of the Constitution",
    "Article 300 a of the Constitution",
    "Article 239 aa of the Constitution",
    "Article 31 c of the Constitution",
    "Articles 226 and 227 of the Constitution",
    "Article 323a and Article 323b of the Constitution",
    "articles into possession and sealed them",
    "articles of material described in sub-item",
]

print("=" * 80)
print("FINAL ARTICLE EXTRACTION TEST")
print("=" * 80)

for i, text in enumerate(final_article_tests, start=1):

    result = extract_articles_final(text)

    print(f"\n{i:02}. Input  : {text}")
    print(f"    Output : {result}")

FINAL ARTICLE EXTRACTION TEST

01. Input  : Article 226 of the Constitution
    Output : 226

02. Input  : Article 136 of the Constitution
    Output : 136

03. Input  : Articles 14 and 19 of the Constitution
    Output : 14

04. Input  : Articles 14, 15 and 16 of the Constitution
    Output : 14

05. Input  : Article 311 2 of the Constitution
    Output : 311

06. Input  : Article 133 1 a of the Constitution
    Output : 133

07. Input  : Article 19 1 g of the Constitution
    Output : 19

08. Input  : Article 16 4a of the Constitution
    Output : 16

09. Input  : Article 300 a of the Constitution
    Output : 300A

10. Input  : Article 239 aa of the Constitution
    Output : 239AA

11. Input  : Article 31 c of the Constitution
    Output : 31C

12. Input  : Articles 226 and 227 of the Constitution
    Output : 226

13. Input  : Article 323a and Article 323b of the Constitution
    Output : 323A,323B

14. Input  : articles into possession and sealed them
    Output : 

15. Input  : a

In [ ]:
# ============================================================
# STEP 44: Final Article Extraction Function - V2
# ============================================================

def extract_articles_final(text):
    """
    Extract Constitutional Article references from legal text.

    Handles:
    - Article 226
    - Article-136
    - Art. 133
    - Articles 14 and 19
    - Articles 14, 15 and 16
    - Article 311 2
    - Article 133 1 a
    - Article 16 4a
    - Article 300 a
    - Article 239 aa
    - Inserted Constitutional Articles
    """

    if pd.isna(text):
        return ""

    text = str(text)

    # --------------------------------------------------------
    # Normalize whitespace
    # --------------------------------------------------------

    text = re.sub(r"\s+", " ", text)

    found_articles = []

    # --------------------------------------------------------
    # Helper function
    # --------------------------------------------------------

    def add_article(raw):

        if not raw:
            return

        normalized = raw.replace(" ", "").upper()

        # Direct canonical match
        if normalized in canonical_articles:
            found_articles.append(normalized)
            return

        # Number + letter
        match = re.match(
            r"^(\d{1,3})([A-Z]{1,2})$",
            normalized
        )

        if match:

            candidate = (
                match.group(1)
                + match.group(2)
            )

            if candidate in canonical_articles:
                found_articles.append(candidate)
                return

            base = match.group(1)

            if base in canonical_articles:
                found_articles.append(base)

    # --------------------------------------------------------
    # Pattern 1: Multiple Article expressions
    #
    # Example:
    # Articles 14 and 19
    # Articles 14, 15 and 16
    # Articles 226, 227 and 32
    # --------------------------------------------------------

    multi_pattern = re.compile(
        r"\barticles?\s+"
        r"(\d{1,3}(?:\s*[a-zA-Z]{1,2})?)"
        r"((?:\s*,\s*|\s+and\s+)"
        r"\d{1,3}(?:\s*[a-zA-Z]{1,2})?)*",
        re.IGNORECASE
    )

    for match in multi_pattern.finditer(text):

        expression = match.group(0)

        # Remove Article / Articles
        expression = re.sub(
            r"^\s*articles?\s+",
            "",
            expression,
            flags=re.IGNORECASE
        )

        # Extract each number + optional letter
        individual_articles = re.findall(
            r"\d{1,3}(?:\s*[a-zA-Z]{1,2})?",
            expression
        )

        for article in individual_articles:
            add_article(article)

    # --------------------------------------------------------
    # Pattern 2: Single Article references
    # --------------------------------------------------------

    single_pattern = re.compile(
        r"\b(?:articles?|arts?)\.?\s*[-:]?\s*"
        r"(\d{1,3}(?:\s*[a-zA-Z]{1,2})?)",
        re.IGNORECASE
    )

    for match in single_pattern.finditer(text):

        add_article(match.group(1))

    # --------------------------------------------------------
    # Remove duplicates while preserving order
    # --------------------------------------------------------

    found_articles = list(
        dict.fromkeys(found_articles)
    )

    return ",".join(found_articles)


print("=" * 80)
print("FINAL ARTICLE EXTRACTION FUNCTION - V2 CREATED")
print("=" * 80)

print("Function: extract_articles_final()")
print("Canonical identifiers:", len(canonical_articles))

FINAL ARTICLE EXTRACTION FUNCTION - V2 CREATED
Function: extract_articles_final()
Canonical identifiers: 461


In [ ]:
# ============================================================
# STEP 44B: Test Final Article Extraction Function
# ============================================================

final_article_tests = [
    "Article 226 of the Constitution",
    "Article 136 of the Constitution",
    "Articles 14 and 19 of the Constitution",
    "Articles 14, 15 and 16 of the Constitution",
    "Article 311 2 of the Constitution",
    "Article 133 1 a of the Constitution",
    "Article 19 1 g of the Constitution",
    "Article 16 4a of the Constitution",
    "Article 300 a of the Constitution",
    "Article 239 aa of the Constitution",
    "Article 31 c of the Constitution",
    "Articles 226 and 227 of the Constitution",
    "Articles 14, 19 and 21 of the Constitution",
    "Articles 132, 133, 134 and 134a of the Constitution",
    "Article 323a and Article 323b of the Constitution",
    "Article 226 and 226 of the Constitution",
    "articles into possession and sealed them",
    "articles of material described in sub-item",
]

print("=" * 80)
print("FINAL ARTICLE EXTRACTION TEST - V2")
print("=" * 80)

for i, text in enumerate(final_article_tests, start=1):

    result = extract_articles_final(text)

    print(f"\n{i:02}. Input  : {text}")
    print(f"    Output : {result}")

FINAL ARTICLE EXTRACTION TEST - V2

01. Input  : Article 226 of the Constitution
    Output : 226

02. Input  : Article 136 of the Constitution
    Output : 136

03. Input  : Articles 14 and 19 of the Constitution
    Output : 14

04. Input  : Articles 14, 15 and 16 of the Constitution
    Output : 14,15

05. Input  : Article 311 2 of the Constitution
    Output : 311

06. Input  : Article 133 1 a of the Constitution
    Output : 133

07. Input  : Article 19 1 g of the Constitution
    Output : 19

08. Input  : Article 16 4a of the Constitution
    Output : 16

09. Input  : Article 300 a of the Constitution
    Output : 300A

10. Input  : Article 239 aa of the Constitution
    Output : 239AA

11. Input  : Article 31 c of the Constitution
    Output : 31C

12. Input  : Articles 226 and 227 of the Constitution
    Output : 226

13. Input  : Articles 14, 19 and 21 of the Constitution
    Output : 14,19

14. Input  : Articles 132, 133, 134 and 134a of the Constitution
    Output : 132,133,

In [ ]:
# ============================================================
# STEP 44C: FINAL ARTICLE EXTRACTION FUNCTION - V3
# ============================================================

def extract_articles_final(text):

    if pd.isna(text):
        return ""

    text = str(text)
    text = re.sub(r"\s+", " ", text)

    found_articles = []

    # --------------------------------------------------------
    # Helper: normalize one Article identifier
    # --------------------------------------------------------

    def normalize_article(raw):

        raw = raw.strip().replace(" ", "").upper()

        # Direct canonical Article
        if raw in canonical_articles:
            return raw

        # Number + letter
        match = re.fullmatch(
            r"(\d{1,3})([A-Z]{1,2})",
            raw
        )

        if match:

            candidate = (
                match.group(1)
                + match.group(2)
            )

            if candidate in canonical_articles:
                return candidate

        return None

    # --------------------------------------------------------
    # STEP 1: Find explicit Article/Articles expressions
    # --------------------------------------------------------

    article_start_pattern = re.compile(
        r"\b(?:articles?|arts?)\.?\s*[-:]?\s*",
        re.IGNORECASE
    )

    matches = list(
        article_start_pattern.finditer(text)
    )

    for i, match in enumerate(matches):

        start = match.end()

        # Stop at the next Article/Art. occurrence
        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = min(
                len(text),
                start + 120
            )

        expression = text[start:end]

        # ----------------------------------------------------
        # Stop expression at legal sentence boundaries
        # ----------------------------------------------------

        expression = re.split(
            r"\b(?:of|under|read|with|and\s+the|which|is|was|were|shall|have|has|had)\b",
            expression,
            maxsplit=1,
            flags=re.IGNORECASE
        )[0]

        # ----------------------------------------------------
        # Extract number / number-letter candidates
        # ----------------------------------------------------

        candidates = re.findall(
            r"\d{1,3}(?:\s*[a-zA-Z]{1,2})?",
            expression
        )

        for candidate in candidates:

            article = normalize_article(candidate)

            if article:
                found_articles.append(article)

    # --------------------------------------------------------
    # STEP 2: Remove duplicates
    # --------------------------------------------------------

    found_articles = list(
        dict.fromkeys(found_articles)
    )

    return ",".join(found_articles)


print("=" * 80)
print("FINAL ARTICLE EXTRACTION FUNCTION - V3 CREATED")
print("=" * 80)

FINAL ARTICLE EXTRACTION FUNCTION - V3 CREATED


In [ ]:
# ============================================================
# STEP 44D: TEST FINAL ARTICLE EXTRACTION - V3
# ============================================================

final_article_tests = [
    "Article 226 of the Constitution",
    "Article 136 of the Constitution",
    "Articles 14 and 19 of the Constitution",
    "Articles 14, 15 and 16 of the Constitution",
    "Article 311 2 of the Constitution",
    "Article 133 1 a of the Constitution",
    "Article 19 1 g of the Constitution",
    "Article 16 4a of the Constitution",
    "Article 300 a of the Constitution",
    "Article 239 aa of the Constitution",
    "Article 31 c of the Constitution",
    "Articles 226 and 227 of the Constitution",
    "Articles 14, 19 and 21 of the Constitution",
    "Articles 132, 133, 134 and 134a of the Constitution",
    "Article 323a and Article 323b of the Constitution",
    "Article 226 and 226 of the Constitution",
    "articles into possession and sealed them",
    "articles of material described in sub-item",
]

print("=" * 80)
print("FINAL ARTICLE EXTRACTION TEST - V3")
print("=" * 80)

for i, text in enumerate(final_article_tests, start=1):

    result = extract_articles_final(text)

    print(f"\n{i:02}. Input  : {text}")
    print(f"    Output : {result}")

FINAL ARTICLE EXTRACTION TEST - V3

01. Input  : Article 226 of the Constitution
    Output : 226

02. Input  : Article 136 of the Constitution
    Output : 136

03. Input  : Articles 14 and 19 of the Constitution
    Output : 19

04. Input  : Articles 14, 15 and 16 of the Constitution
    Output : 14,16

05. Input  : Article 311 2 of the Constitution
    Output : 311,2

06. Input  : Article 133 1 a of the Constitution
    Output : 133

07. Input  : Article 19 1 g of the Constitution
    Output : 19

08. Input  : Article 16 4a of the Constitution
    Output : 16

09. Input  : Article 300 a of the Constitution
    Output : 300A

10. Input  : Article 239 aa of the Constitution
    Output : 239AA

11. Input  : Article 31 c of the Constitution
    Output : 31C

12. Input  : Articles 226 and 227 of the Constitution
    Output : 227

13. Input  : Articles 14, 19 and 21 of the Constitution
    Output : 14,21

14. Input  : Articles 132, 133, 134 and 134a of the Constitution
    Output : 132,13

In [ ]:
# ============================================================
# STEP 45: Create Robust Article-Based Features
# ============================================================

print("=" * 80)
print("CREATING ROBUST ARTICLE FEATURES")
print("=" * 80)

# Use the best available Article extraction for reference detection
article_temp = model_df["judgement"].apply(
    extract_articles_v4
)

# ------------------------------------------------------------
# Feature 1: Whether the judgment contains Article references
# ------------------------------------------------------------

model_df_final = model_df.copy()

model_df_final["has_article_reference"] = (
    article_temp != ""
).astype(int)

# ------------------------------------------------------------
# Feature 2: Number of extracted Article identifiers
# ------------------------------------------------------------

model_df_final["article_count"] = article_temp.apply(
    lambda x: len(x.split(",")) if x else 0
)

print("Article-based features created successfully.")

print("\n" + "=" * 80)
print("FEATURE SUMMARY")
print("=" * 80)

print(
    model_df_final[
        ["has_article_reference", "article_count"]
    ].describe()
)

print("\nHas Article Reference:")
print(
    model_df_final["has_article_reference"]
    .value_counts()
    .sort_index()
)

print("\nArticle Count Distribution:")
print(
    model_df_final["article_count"]
    .value_counts()
    .sort_index()
    .head(20)
)

print("\nSample:")
display(
    model_df_final[
        [
            "judgement",
            "has_article_reference",
            "article_count",
            "label"
        ]
    ].head(10)
)

CREATING ROBUST ARTICLE FEATURES
Article-based features created successfully.

FEATURE SUMMARY
       has_article_reference  article_count
count           42169.000000   42169.000000
mean                0.348242       1.291802
std                 0.476419       2.818047
min                 0.000000       0.000000
25%                 0.000000       0.000000
50%                 0.000000       0.000000
75%                 1.000000       1.000000
max                 1.000000      48.000000

Has Article Reference:
has_article_reference
0    27484
1    14685
Name: count, dtype: int64

Article Count Distribution:
article_count
0     27484
1      4864
2      2788
3      1729
4      1318
5       931
6       750
7       514
8       421
9       311
10      209
11      196
12      145
13      100
14       94
15       55
16       58
17       39
18       36
19       22
Name: count, dtype: int64

Sample:


,judgement,has_article_reference,article_count,label
0,thus spake benjamin franklin in his letter of ...,1,2,1
1,difficult to share the view taken by the high...,1,2,1
2,j u d g m e n t the following judgement of the...,1,1,1
3,custodian first stated that the property was n...,0,0,1
4,this appeal by special leave is against the ju...,0,0,0
5,"as the arguments proceeded, it became clear to...",1,1,0
6,"j u d g m e n t harjit singh bedi, j. this app...",0,0,0
7,this appeal by special leave is directed again...,0,0,1
8,this is an appeal against an order passed by t...,0,0,1
9,stablish the plea of innocence examined two pe...,0,0,0


In [ ]:
# ============================================================
# STEP 46: Inspect Case Metadata Features
# ============================================================

print("=" * 80)
print("CASE METADATA FEATURE INSPECTION")
print("=" * 80)

print("\nCASE CATEGORY")
print("-" * 80)

print(
    "Unique categories:",
    model_df_final["case_category"].nunique()
)

print(
    model_df_final["case_category"]
    .value_counts(dropna=False)
    .head(20)
)

print("\nCASE TYPE")
print("-" * 80)

print(
    "Unique case types:",
    model_df_final["case_type"].nunique()
)

print(
    model_df_final["case_type"]
    .value_counts(dropna=False)
    .head(30)
)

print("\nMissing values:")
print(
    model_df_final[
        ["case_category", "case_type"]
    ].isnull().sum()
)

CASE METADATA FEATURE INSPECTION

CASE CATEGORY
--------------------------------------------------------------------------------
Unique categories: 2
case_category
civil       28124
criminal     7328
NaN          6717
Name: count, dtype: int64

CASE TYPE
--------------------------------------------------------------------------------
Unique case types: 11
case_type
appeal                    23549
writ petition             13778
special leave petition     2899
election petition           769
contempt petition           594
review petition             475
transfer petition            57
tax reference case           23
transferred case             17
arbitration appeal            7
curative petition             1
Name: count, dtype: int64

Missing values:
case_category    6717
case_type           0
dtype: int64


In [ ]:
# ============================================================
# STEP 47: Prepare Metadata Features
# ============================================================

print("=" * 80)
print("PREPARING CASE METADATA FEATURES")
print("=" * 80)

# Create a copy for modeling
modeling_df = model_df_final.copy()

# ------------------------------------------------------------
# Handle missing case category
# ------------------------------------------------------------

modeling_df["case_category"] = (
    modeling_df["case_category"]
    .fillna("unknown")
)

# ------------------------------------------------------------
# Verify categorical features
# ------------------------------------------------------------

print("Case Category Values:")
print(
    modeling_df["case_category"]
    .value_counts()
)

print("\nCase Type Values:")
print(
    modeling_df["case_type"]
    .value_counts()
)

print("\nMissing Values:")
print(
    modeling_df[
        ["case_category", "case_type"]
    ].isnull().sum()
)

print("\nMetadata preparation completed successfully.")

PREPARING CASE METADATA FEATURES
Case Category Values:
case_category
civil       28124
criminal     7328
unknown      6717
Name: count, dtype: int64

Case Type Values:
case_type
appeal                    23549
writ petition             13778
special leave petition     2899
election petition           769
contempt petition           594
review petition             475
transfer petition            57
tax reference case           23
transferred case             17
arbitration appeal            7
curative petition             1
Name: count, dtype: int64

Missing Values:
case_category    0
case_type        0
dtype: int64

Metadata preparation completed successfully.


In [ ]:
# ============================================================
# STEP 48: Create Fixed Train-Test Split
# ============================================================

from sklearn.model_selection import train_test_split

print("=" * 80)
print("CREATING FIXED TRAIN-TEST SPLIT")
print("=" * 80)

# Split indices instead of the complete dataset
# so the exact same split can be reused for every experiment.

train_indices, test_indices = train_test_split(
    modeling_df.index,
    test_size=0.20,
    random_state=42,
    stratify=modeling_df["label"]
)

# Create train and test datasets
train_df = modeling_df.loc[train_indices].copy()
test_df = modeling_df.loc[test_indices].copy()

# Reset indices
train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

print(f"Total Samples : {len(modeling_df):,}")
print(f"Training Samples: {len(train_df):,}")
print(f"Testing Samples : {len(test_df):,}")

print("\nTraining Label Distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nTesting Label Distribution:")
print(test_df["label"].value_counts().sort_index())

CREATING FIXED TRAIN-TEST SPLIT
Total Samples : 42,169
Training Samples: 33,735
Testing Samples : 8,434

Training Label Distribution:
label
0    17942
1    15793
Name: count, dtype: int64

Testing Label Distribution:
label
0    4486
1    3948
Name: count, dtype: int64


In [ ]:
# ============================================================
# STEP 49: Create Baseline TF-IDF Features
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

print("=" * 80)
print("CREATING BASELINE TF-IDF FEATURES")
print("=" * 80)

# Initialize TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=50000
)

# ------------------------------------------------------------
# Fit ONLY on training judgments
# ------------------------------------------------------------

X_train_text = tfidf_vectorizer.fit_transform(
    train_df["cleaned_judgement"]
)

# ------------------------------------------------------------
# Transform test judgments using the training vocabulary
# ------------------------------------------------------------

X_test_text = tfidf_vectorizer.transform(
    test_df["cleaned_judgement"]
)

print("TF-IDF vectorization completed successfully.")

print("\nTraining Feature Matrix Shape:")
print(X_train_text.shape)

print("\nTesting Feature Matrix Shape:")
print(X_test_text.shape)

print("\nVocabulary Size:")
print(len(tfidf_vectorizer.vocabulary_))

print("\nFirst 20 TF-IDF Features:")
print(
    tfidf_vectorizer.get_feature_names_out()[:20]
)

CREATING BASELINE TF-IDF FEATURES
TF-IDF vectorization completed successfully.

Training Feature Matrix Shape:
(33735, 50000)

Testing Feature Matrix Shape:
(8434, 50000)

Vocabulary Size:
50000

First 20 TF-IDF Features:
['000' '0000' '00000' '000000' '0001' '0007' '001' '002' '003' '0030'
 '004' '0045' '005' '006' '007' '008' '009' '00a' '00p' '00pm']


In [ ]:
# ============================================================
# STEP 50: Prepare Target Variables
# ============================================================

print("=" * 80)
print("PREPARING TARGET VARIABLES")
print("=" * 80)

# Training labels
y_train = train_df["label"].values

# Testing labels
y_test = test_df["label"].values

print("Target variables prepared successfully.")

print("\nTraining Target Shape:")
print(y_train.shape)

print("\nTesting Target Shape:")
print(y_test.shape)

print("\nTraining Label Distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nTesting Label Distribution:")
print(pd.Series(y_test).value_counts().sort_index())

PREPARING TARGET VARIABLES
Target variables prepared successfully.

Training Target Shape:
(33735,)

Testing Target Shape:
(8434,)

Training Label Distribution:
0    17942
1    15793
Name: count, dtype: int64

Testing Label Distribution:
0    4486
1    3948
Name: count, dtype: int64


In [ ]:
# ============================================================
# STEP 51: Train Baseline Models
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
import time

print("=" * 80)
print("BASELINE MODEL TRAINING")
print("=" * 80)

# ------------------------------------------------------------
# Store results
# ------------------------------------------------------------

baseline_results = []

# ============================================================
# 1. LOGISTIC REGRESSION
# ============================================================

print("\n" + "-" * 80)
print("1. LOGISTIC REGRESSION")
print("-" * 80)

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

start_time = time.time()

logistic_model.fit(
    X_train_text,
    y_train
)

logistic_train_time = time.time() - start_time

start_time = time.time()

logistic_predictions = logistic_model.predict(
    X_test_text
)

logistic_prediction_time = time.time() - start_time

logistic_accuracy = accuracy_score(
    y_test,
    logistic_predictions
)

logistic_precision = precision_score(
    y_test,
    logistic_predictions
)

logistic_recall = recall_score(
    y_test,
    logistic_predictions
)

logistic_f1 = f1_score(
    y_test,
    logistic_predictions
)

print(f"Training Time   : {logistic_train_time:.2f} seconds")
print(f"Prediction Time : {logistic_prediction_time:.4f} seconds")
print(f"Accuracy        : {logistic_accuracy:.4f}")
print(f"Precision       : {logistic_precision:.4f}")
print(f"Recall          : {logistic_recall:.4f}")
print(f"F1-Score        : {logistic_f1:.4f}")


# ============================================================
# 2. LINEAR SVM
# ============================================================

print("\n" + "-" * 80)
print("2. LINEAR SVM")
print("-" * 80)

svm_model = LinearSVC(
    random_state=42
)

start_time = time.time()

svm_model.fit(
    X_train_text,
    y_train
)

svm_train_time = time.time() - start_time

start_time = time.time()

svm_predictions = svm_model.predict(
    X_test_text
)

svm_prediction_time = time.time() - start_time

svm_accuracy = accuracy_score(
    y_test,
    svm_predictions
)

svm_precision = precision_score(
    y_test,
    svm_predictions
)

svm_recall = recall_score(
    y_test,
    svm_predictions
)

svm_f1 = f1_score(
    y_test,
    svm_predictions
)

print(f"Training Time   : {svm_train_time:.2f} seconds")
print(f"Prediction Time : {svm_prediction_time:.4f} seconds")
print(f"Accuracy        : {svm_accuracy:.4f}")
print(f"Precision       : {svm_precision:.4f}")
print(f"Recall          : {svm_recall:.4f}")
print(f"F1-Score        : {svm_f1:.4f}")


# ============================================================
# 3. RANDOM FOREST
# ============================================================

print("\n" + "-" * 80)
print("3. RANDOM FOREST")
print("-" * 80)

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

random_forest_model.fit(
    X_train_text,
    y_train
)

rf_train_time = time.time() - start_time

start_time = time.time()

rf_predictions = random_forest_model.predict(
    X_test_text
)

rf_prediction_time = time.time() - start_time

rf_accuracy = accuracy_score(
    y_test,
    rf_predictions
)

rf_precision = precision_score(
    y_test,
    rf_predictions
)

rf_recall = recall_score(
    y_test,
    rf_predictions
)

rf_f1 = f1_score(
    y_test,
    rf_predictions
)

print(f"Training Time   : {rf_train_time:.2f} seconds")
print(f"Prediction Time : {rf_prediction_time:.4f} seconds")
print(f"Accuracy        : {rf_accuracy:.4f}")
print(f"Precision       : {rf_precision:.4f}")
print(f"Recall          : {rf_recall:.4f}")
print(f"F1-Score        : {rf_f1:.4f}")


# ============================================================
# CREATE BASELINE RESULTS TABLE
# ============================================================

baseline_results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": logistic_accuracy,
        "Precision": logistic_precision,
        "Recall": logistic_recall,
        "F1-Score": logistic_f1,
        "Training Time": logistic_train_time,
        "Prediction Time": logistic_prediction_time
    },
    {
        "Model": "Linear SVM",
        "Accuracy": svm_accuracy,
        "Precision": svm_precision,
        "Recall": svm_recall,
        "F1-Score": svm_f1,
        "Training Time": svm_train_time,
        "Prediction Time": svm_prediction_time
    },
    {
        "Model": "Random Forest",
        "Accuracy": rf_accuracy,
        "Precision": rf_precision,
        "Recall": rf_recall,
        "F1-Score": rf_f1,
        "Training Time": rf_train_time,
        "Prediction Time": rf_prediction_time
    }
])

print("\n" + "=" * 80)
print("BASELINE MODEL COMPARISON")
print("=" * 80)

display(
    baseline_results.round(4)
)

BASELINE MODEL TRAINING

--------------------------------------------------------------------------------
1. LOGISTIC REGRESSION
--------------------------------------------------------------------------------
Training Time   : 16.94 seconds
Prediction Time : 0.0294 seconds
Accuracy        : 0.7534
Precision       : 0.7607
Recall          : 0.6902
F1-Score        : 0.7238

--------------------------------------------------------------------------------
2. LINEAR SVM
--------------------------------------------------------------------------------
Training Time   : 4.58 seconds
Prediction Time : 0.0118 seconds
Accuracy        : 0.7884
Precision       : 0.7822
Recall          : 0.7594
F1-Score        : 0.7706

--------------------------------------------------------------------------------
3. RANDOM FOREST
--------------------------------------------------------------------------------
Training Time   : 157.76 seconds
Prediction Time : 1.5703 seconds
Accuracy        : 0.8255
Precision    

,Model,Accuracy,Precision,Recall,F1-Score,Training Time,Prediction Time
0,Logistic Regression,0.7534,0.7607,0.6902,0.7238,16.9416,0.0294
1,Linear SVM,0.7884,0.7822,0.7594,0.7706,4.5755,0.0118
2,Random Forest,0.8255,0.8483,0.7637,0.8038,157.7621,1.5703


In [ ]:
# ============================================================
# STEP 52: Save Baseline Results
# ============================================================

baseline_results.to_csv(
    "/content/drive/MyDrive/Legal_Project/Dataset/baseline_model_results.csv",
    index=False
)

print("=" * 80)
print("BASELINE RESULTS SAVED SUCCESSFULLY")
print("=" * 80)

print(
    "Location: /content/drive/MyDrive/Legal_Project/Dataset/"
    "baseline_model_results.csv"
)

display(baseline_results.round(4))

BASELINE RESULTS SAVED SUCCESSFULLY
Location: /content/drive/MyDrive/Legal_Project/Dataset/baseline_model_results.csv


,Model,Accuracy,Precision,Recall,F1-Score,Training Time,Prediction Time
0,Logistic Regression,0.7534,0.7607,0.6902,0.7238,16.9416,0.0294
1,Linear SVM,0.7884,0.7822,0.7594,0.7706,4.5755,0.0118
2,Random Forest,0.8255,0.8483,0.7637,0.8038,157.7621,1.5703


In [ ]:
# ============================================================
# RECOVERY STEP 1: Reload Preprocessed Dataset
# ============================================================

import pandas as pd
import numpy as np

dataset_path = (
    "/content/drive/MyDrive/Legal_Project/Dataset/"
    "case_files_preprocessed.csv"
)

model_df = pd.read_csv(dataset_path)

print("=" * 80)
print("PREPROCESSED DATASET RESTORED")
print("=" * 80)

print("Shape:", model_df.shape)

print("\nColumns:")
print(model_df.columns.tolist())

print("\nLabel Distribution:")
print(model_df["label"].value_counts().sort_index())

PREPROCESSED DATASET RESTORED
Shape: (42169, 9)

Columns:
['case_category', 'case_type', 'case_info', 'judgement', 'label', 'char_count', 'word_count', 'sentence_count', 'cleaned_judgement']

Label Distribution:
label
0    22428
1    19741
Name: count, dtype: int64


In [ ]:
# ============================================================
# RECOVERY STEP 2A: Check Article Extractor
# ============================================================

try:
    print(extract_articles_v4)
    print("\nextract_articles_v4 is available.")
except NameError:
    print("extract_articles_v4 is NOT available.")

extract_articles_v4 is NOT available.


In [ ]:
# ============================================================
# RECOVERY STEP 2: Restore Canonical Article Extractor
# ============================================================

import re

# ------------------------------------------------------------
# Canonical constitutional Article identifiers
# ------------------------------------------------------------

canonical_articles = set("""
1 2 2A 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20
21 21A 22 23 24 25 26 27 28 29 30 31 31A 31B 31C 32 33
34 35 36 37 38 39 39A 40 41 42 43 43A 44 45 46 47 48
48A 49 50 51 51A 52 53 54 55 56 57 58 59 60 61 62 63
64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81
82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98
99 100 101 102 103 104 105 106 107 108 109 110 111 112
113 114 115 116 117 118 119 120 121 122 123 124 125 126
127 128 129 130 131 132 133 134 134A 135 136 137 138 139
140 141 142 143 144 145 146 147 148 149 150 151 152 153
154 155 156 157 158 159 160 161 162 163 164 165 166 167
168 169 170 171 172 173 174 175 176 177 178 179 180 181
182 183 184 185 186 187 188 189 190 191 192 193 194 195
196 197 198 199 200 201 202 203 204 205 206 207 208 209
210 211 212 213 214 215 216 217 218 219 220 221 222 223
224 224A 225 226 227 228 229 230 231 232 233 234 235 236
237 238 239 239A 239AA 240 241 242 243 243A 243B 243C 243D
243E 243F 243G 243H 243I 243J 243K 243L 243M 243N 243O 243P
243Q 243R 243S 243T 243U 243V 243W 243X 243Y 243Z 243ZA 243ZB
243ZC 243ZD 243ZE 243ZF 243ZG 244 244A 245 246 247 248 249
250 251 252 253 254 255 256 257 258 259 260 261 262 263
264 265 266 267 268 269 270 271 272 273 274 275 276 277
278 279 280 281 282 283 284 285 286 287 288 289 290 291
292 293 294 295 296 297 298 299 300 300A 301 302 303 304
305 306 307 308 309 310 311 312 313 314 315 316 317 318
319 320 321 322 323 323A 323B 324 325 326 327 328 329
330 330A 331 332 332A 333 334 334A 335 336 337 338 339 340
341 342 342A 343 344 345 346 347 348 349 350 351 352 353
354 355 356 357 358 359 360 361 362 363 364 365 366 367
368 369 370 371 371A 371B 371C 371D 371E 371F 371G 371H
371I 371J 372 372A 373 374 375 376 377 378 378A 379 380
381 382 383 384 385 386 387 388 389 390 391 392 393 394
394A 395 396
""".split())

print("=" * 80)
print("CANONICAL ARTICLE SET RESTORED")
print("=" * 80)

print("Canonical identifiers:", len(canonical_articles))


# ------------------------------------------------------------
# Normalize Article identifiers
# ------------------------------------------------------------

def normalize_article_identifier(number, letter=None):
    """
    Convert Article references such as:

        300 a   -> 300A
        239 aa  -> 239AA
        31 c    -> 31C
        134a    -> 134A

    Only return the identifier if it belongs to the
    canonical constitutional Article set.
    """

    number = str(number).strip()

    if letter:
        letter = str(letter).strip().upper()
        candidate = number + letter
    else:
        candidate = number

    if candidate in canonical_articles:
        return candidate

    return None


# ------------------------------------------------------------
# Final Article extraction function
# ------------------------------------------------------------

def extract_articles_v4(text):
    """
    Extract recognizable constitutional Article references.

    Handles:
    - Article 226
    - Article 31C
    - Article 300 a
    - Article 239 aa
    - Articles 14 and 19
    - Articles 132, 133, 134 and 134a
    - Article-136
    - Art. 133

    Avoids ordinary uses of the word 'article'.
    """

    if pd.isna(text):
        return ""

    text = str(text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    found = []

    # --------------------------------------------------------
    # Patterns beginning with Article / Articles / Art.
    # --------------------------------------------------------

    pattern = re.compile(
        r"\barticles?\s+"
        r"(\d{1,3})"
        r"(?:\s*[-]?\s*([a-z]{1,2}))?"
        r"(?=\s|,|\.|\)|$)",
        flags=re.IGNORECASE
    )

    for match in pattern.finditer(text):

        number = match.group(1)
        letter = match.group(2)

        candidate = normalize_article_identifier(
            number,
            letter
        )

        if candidate:
            found.append(candidate)

        # ----------------------------------------------------
        # Look ahead for additional Articles in expressions
        # such as:
        #
        # Articles 14 and 19
        # Articles 14, 15 and 16
        # Articles 132, 133, 134 and 134a
        # ----------------------------------------------------

        remainder = text[match.end():]

        # Stop the continuation search at a reasonable boundary
        continuation = re.match(
            r"((?:\s*(?:,|and)\s*)"
            r"(?:\d{1,3})(?:\s*[a-z]{1,2})?)+",
            remainder,
            flags=re.IGNORECASE
        )

        if continuation:

            pieces = re.findall(
                r"\d{1,3}(?:\s*[a-z]{1,2})?",
                continuation.group(0),
                flags=re.IGNORECASE
            )

            for piece in pieces:

                piece = piece.strip()

                number_match = re.match(
                    r"(\d{1,3})",
                    piece
                )

                if not number_match:
                    continue

                number = number_match.group(1)

                letter_match = re.search(
                    r"([a-z]{1,2})$",
                    piece,
                    flags=re.IGNORECASE
                )

                letter = (
                    letter_match.group(1)
                    if letter_match
                    else None
                )

                candidate = normalize_article_identifier(
                    number,
                    letter
                )

                if candidate:
                    found.append(candidate)

    # --------------------------------------------------------
    # Remove duplicates while preserving order
    # --------------------------------------------------------

    unique_found = list(dict.fromkeys(found))

    return ",".join(unique_found)


print("\nFunction restored successfully:")
print("extract_articles_v4")

CANONICAL ARTICLE SET RESTORED
Canonical identifiers: 463

Function restored successfully:
extract_articles_v4


In [ ]:
# ============================================================
# RECOVERY STEP 3: Validate Restored Article Extractor
# ============================================================

test_cases = [
    "Article 226 of the Constitution",
    "Article 136 of the Constitution",
    "Articles 14 and 19 of the Constitution",
    "Articles 14, 15 and 16 of the Constitution",
    "Article 311 2 of the Constitution",
    "Article 133 1 a of the Constitution",
    "Article 19 1 g of the Constitution",
    "Article 16 4a of the Constitution",
    "Article 300 a of the Constitution",
    "Article 239 aa of the Constitution",
    "Article 31 c of the Constitution",
    "Articles 226 and 227 of the Constitution",
    "Articles 132, 133, 134 and 134a of the Constitution",
    "Article 323a and Article 323b of the Constitution",
    "articles into possession and sealed them",
    "articles of material described in sub-item"
]

print("=" * 80)
print("RESTORED ARTICLE EXTRACTOR VALIDATION")
print("=" * 80)

for i, text in enumerate(test_cases, 1):
    result = extract_articles_v4(text)

    print(f"{i:02d}. Input : {text}")
    print(f"    Output: {result}")
    print()

RESTORED ARTICLE EXTRACTOR VALIDATION
01. Input : Article 226 of the Constitution
    Output: 

02. Input : Article 136 of the Constitution
    Output: 

03. Input : Articles 14 and 19 of the Constitution
    Output: 14

04. Input : Articles 14, 15 and 16 of the Constitution
    Output: 14

05. Input : Article 311 2 of the Constitution
    Output: 311

06. Input : Article 133 1 a of the Constitution
    Output: 133

07. Input : Article 19 1 g of the Constitution
    Output: 19

08. Input : Article 16 4a of the Constitution
    Output: 16

09. Input : Article 300 a of the Constitution
    Output: 300A

10. Input : Article 239 aa of the Constitution
    Output: 239AA

11. Input : Article 31 c of the Constitution
    Output: 31C

12. Input : Articles 226 and 227 of the Constitution
    Output: 226

13. Input : Articles 132, 133, 134 and 134a of the Constitution
    Output: 132,133

14. Input : Article 323a and Article 323b of the Constitution
    Output: 323A,323B

15. Input : articles in

In [ ]:
# ============================================================
# RECOVERY STEP 3B: Check Saved Modeling Files
# ============================================================

import os

dataset_folder = "/content/drive/MyDrive/Legal_Project/Dataset"

print("=" * 80)
print("CHECKING SAVED DATASET FILES")
print("=" * 80)

for filename in sorted(os.listdir(dataset_folder)):
    print(filename)

CHECKING SAVED DATASET FILES
baseline_model_results.csv
case_files_preprocessed.csv
case_files_total.csv


In [ ]:
# ============================================================
# RECOVERY STEP 4: Recreate Modeling Dataset
# ============================================================

print("=" * 80)
print("RECREATING MODELING DATASET")
print("=" * 80)

# Start from the saved preprocessed dataset
modeling_df = model_df.copy()

# ------------------------------------------------------------
# Handle missing case category
# ------------------------------------------------------------

modeling_df["case_category"] = (
    modeling_df["case_category"]
    .fillna("unknown")
)

# ------------------------------------------------------------
# Verify required columns
# ------------------------------------------------------------

required_columns = [
    "case_category",
    "case_type",
    "judgement",
    "cleaned_judgement",
    "label"
]

missing_required = [
    col for col in required_columns
    if col not in modeling_df.columns
]

if missing_required:
    raise ValueError(
        f"Missing required columns: {missing_required}"
    )

print("Required columns verified.")

print("\nDataset Shape:")
print(modeling_df.shape)

print("\nMissing Values:")
print(
    modeling_df[
        required_columns
    ].isnull().sum()
)

print("\nLabel Distribution:")
print(
    modeling_df["label"]
    .value_counts()
    .sort_index()
)

print("\nModeling dataset recreated successfully.")

RECREATING MODELING DATASET
Required columns verified.

Dataset Shape:
(42169, 9)

Missing Values:
case_category        0
case_type            0
judgement            0
cleaned_judgement    0
label                0
dtype: int64

Label Distribution:
label
0    22428
1    19741
Name: count, dtype: int64

Modeling dataset recreated successfully.


In [ ]:
# ============================================================
# RECOVERY STEP 5: Recreate Fixed Train-Test Split
# ============================================================

from sklearn.model_selection import train_test_split

print("=" * 80)
print("RECREATING FIXED TRAIN-TEST SPLIT")
print("=" * 80)

train_indices, test_indices = train_test_split(
    modeling_df.index,
    test_size=0.20,
    random_state=42,
    stratify=modeling_df["label"]
)

train_df = modeling_df.loc[train_indices].copy()
test_df = modeling_df.loc[test_indices].copy()

train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

print(f"Total Samples    : {len(modeling_df):,}")
print(f"Training Samples : {len(train_df):,}")
print(f"Testing Samples  : {len(test_df):,}")

print("\nTraining Label Distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nTesting Label Distribution:")
print(test_df["label"].value_counts().sort_index())

RECREATING FIXED TRAIN-TEST SPLIT
Total Samples    : 42,169
Training Samples : 33,735
Testing Samples  : 8,434

Training Label Distribution:
label
0    17942
1    15793
Name: count, dtype: int64

Testing Label Distribution:
label
0    4486
1    3948
Name: count, dtype: int64


In [ ]:
# ============================================================
# RECOVERY STEP 6: Recreate TF-IDF Features
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

print("=" * 80)
print("RECREATING TF-IDF FEATURES")
print("=" * 80)

tfidf_vectorizer = TfidfVectorizer(
    max_features=50000
)

# Fit ONLY on training text
X_train_text = tfidf_vectorizer.fit_transform(
    train_df["cleaned_judgement"]
)

# Transform test text using training vocabulary
X_test_text = tfidf_vectorizer.transform(
    test_df["cleaned_judgement"]
)

print("TF-IDF features recreated successfully.")

print("\nTraining Feature Matrix:")
print(X_train_text.shape)

print("\nTesting Feature Matrix:")
print(X_test_text.shape)

print("\nVocabulary Size:")
print(len(tfidf_vectorizer.vocabulary_))

RECREATING TF-IDF FEATURES
TF-IDF features recreated successfully.

Training Feature Matrix:
(33735, 50000)

Testing Feature Matrix:
(8434, 50000)

Vocabulary Size:
50000


In [ ]:
# ============================================================
# RECOVERY STEP 7: Restore Target Variables
# ============================================================

y_train = train_df["label"].values
y_test = test_df["label"].values

print("=" * 80)
print("TARGET VARIABLES RESTORED")
print("=" * 80)

print("Training Target Shape:", y_train.shape)
print("Testing Target Shape :", y_test.shape)

print("\nTraining Label Distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nTesting Label Distribution:")
print(pd.Series(y_test).value_counts().sort_index())

TARGET VARIABLES RESTORED
Training Target Shape: (33735,)
Testing Target Shape : (8434,)

Training Label Distribution:
0    17942
1    15793
Name: count, dtype: int64

Testing Label Distribution:
0    4486
1    3948
Name: count, dtype: int64


In [ ]:
# ============================================================
# EXPERIMENT 2: TF-IDF + CASE METADATA
# STEP 8: One-Hot Encode Case Metadata
# ============================================================

from sklearn.preprocessing import OneHotEncoder

print("=" * 80)
print("ENCODING CASE METADATA")
print("=" * 80)

categorical_columns = [
    "case_category",
    "case_type"
]

metadata_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

# Fit ONLY on training metadata
X_train_metadata = metadata_encoder.fit_transform(
    train_df[categorical_columns]
)

# Transform test metadata
X_test_metadata = metadata_encoder.transform(
    test_df[categorical_columns]
)

print("Metadata encoding completed successfully.")

print("\nTraining Metadata Matrix Shape:")
print(X_train_metadata.shape)

print("\nTesting Metadata Matrix Shape:")
print(X_test_metadata.shape)

print("\nEncoded Features:")
print(
    metadata_encoder.get_feature_names_out(
        categorical_columns
    )
)

ENCODING CASE METADATA
Metadata encoding completed successfully.

Training Metadata Matrix Shape:
(33735, 14)

Testing Metadata Matrix Shape:
(8434, 14)

Encoded Features:
['case_category_civil' 'case_category_criminal' 'case_category_unknown'
 'case_type_appeal' 'case_type_arbitration appeal'
 'case_type_contempt petition' 'case_type_curative petition'
 'case_type_election petition' 'case_type_review petition'
 'case_type_special leave petition' 'case_type_tax reference case'
 'case_type_transfer petition' 'case_type_transferred case'
 'case_type_writ petition']


In [ ]:
# ============================================================
# EXPERIMENT 2: TF-IDF + CASE METADATA
# STEP 9: Combine Feature Matrices
# ============================================================

from scipy.sparse import hstack

print("=" * 80)
print("COMBINING TF-IDF + CASE METADATA")
print("=" * 80)

# Combine text and metadata horizontally
X_train_metadata_combined = hstack(
    [
        X_train_text,
        X_train_metadata
    ],
    format="csr"
)

X_test_metadata_combined = hstack(
    [
        X_test_text,
        X_test_metadata
    ],
    format="csr"
)

print("Feature combination completed successfully.")

print("\nTraining Combined Matrix:")
print(X_train_metadata_combined.shape)

print("\nTesting Combined Matrix:")
print(X_test_metadata_combined.shape)

print("\nFeature Count:")
print(
    X_train_metadata_combined.shape[1]
)

COMBINING TF-IDF + CASE METADATA
Feature combination completed successfully.

Training Combined Matrix:
(33735, 50014)

Testing Combined Matrix:
(8434, 50014)

Feature Count:
50014


In [ ]:
# ============================================================
# EXPERIMENT 2
# STEP 10: Train Models with TF-IDF + Case Metadata
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import time

print("=" * 80)
print("EXPERIMENT 2: TF-IDF + CASE METADATA")
print("=" * 80)

metadata_results_list = []

# ============================================================
# 1. LOGISTIC REGRESSION
# ============================================================

print("\n" + "-" * 80)
print("1. LOGISTIC REGRESSION")
print("-" * 80)

metadata_logistic = LogisticRegression(
    max_iter=1000,
    random_state=42
)

start_time = time.time()

metadata_logistic.fit(
    X_train_metadata_combined,
    y_train
)

metadata_logistic_train_time = time.time() - start_time

start_time = time.time()

metadata_logistic_predictions = (
    metadata_logistic.predict(
        X_test_metadata_combined
    )
)

metadata_logistic_prediction_time = (
    time.time() - start_time
)

metadata_logistic_accuracy = accuracy_score(
    y_test,
    metadata_logistic_predictions
)

metadata_logistic_precision = precision_score(
    y_test,
    metadata_logistic_predictions
)

metadata_logistic_recall = recall_score(
    y_test,
    metadata_logistic_predictions
)

metadata_logistic_f1 = f1_score(
    y_test,
    metadata_logistic_predictions
)

print(f"Training Time   : {metadata_logistic_train_time:.2f} seconds")
print(f"Prediction Time : {metadata_logistic_prediction_time:.4f} seconds")
print(f"Accuracy        : {metadata_logistic_accuracy:.4f}")
print(f"Precision       : {metadata_logistic_precision:.4f}")
print(f"Recall          : {metadata_logistic_recall:.4f}")
print(f"F1-Score        : {metadata_logistic_f1:.4f}")


# ============================================================
# 2. LINEAR SVM
# ============================================================

print("\n" + "-" * 80)
print("2. LINEAR SVM")
print("-" * 80)

metadata_svm = LinearSVC(
    random_state=42
)

start_time = time.time()

metadata_svm.fit(
    X_train_metadata_combined,
    y_train
)

metadata_svm_train_time = time.time() - start_time

start_time = time.time()

metadata_svm_predictions = (
    metadata_svm.predict(
        X_test_metadata_combined
    )
)

metadata_svm_prediction_time = (
    time.time() - start_time
)

metadata_svm_accuracy = accuracy_score(
    y_test,
    metadata_svm_predictions
)

metadata_svm_precision = precision_score(
    y_test,
    metadata_svm_predictions
)

metadata_svm_recall = recall_score(
    y_test,
    metadata_svm_predictions
)

metadata_svm_f1 = f1_score(
    y_test,
    metadata_svm_predictions
)

print(f"Training Time   : {metadata_svm_train_time:.2f} seconds")
print(f"Prediction Time : {metadata_svm_prediction_time:.4f} seconds")
print(f"Accuracy        : {metadata_svm_accuracy:.4f}")
print(f"Precision       : {metadata_svm_precision:.4f}")
print(f"Recall          : {metadata_svm_recall:.4f}")
print(f"F1-Score        : {metadata_svm_f1:.4f}")


# ============================================================
# 3. RANDOM FOREST
# ============================================================

print("\n" + "-" * 80)
print("3. RANDOM FOREST")
print("-" * 80)

metadata_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

metadata_rf.fit(
    X_train_metadata_combined,
    y_train
)

metadata_rf_train_time = time.time() - start_time

start_time = time.time()

metadata_rf_predictions = (
    metadata_rf.predict(
        X_test_metadata_combined
    )
)

metadata_rf_prediction_time = (
    time.time() - start_time
)

metadata_rf_accuracy = accuracy_score(
    y_test,
    metadata_rf_predictions
)

metadata_rf_precision = precision_score(
    y_test,
    metadata_rf_predictions
)

metadata_rf_recall = recall_score(
    y_test,
    metadata_rf_predictions
)

metadata_rf_f1 = f1_score(
    y_test,
    metadata_rf_predictions
)

print(f"Training Time   : {metadata_rf_train_time:.2f} seconds")
print(f"Prediction Time : {metadata_rf_prediction_time:.4f} seconds")
print(f"Accuracy        : {metadata_rf_accuracy:.4f}")
print(f"Precision       : {metadata_rf_precision:.4f}")
print(f"Recall          : {metadata_rf_recall:.4f}")
print(f"F1-Score        : {metadata_rf_f1:.4f}")


# ============================================================
# CREATE RESULTS TABLE
# ============================================================

metadata_results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": metadata_logistic_accuracy,
        "Precision": metadata_logistic_precision,
        "Recall": metadata_logistic_recall,
        "F1-Score": metadata_logistic_f1,
        "Training Time": metadata_logistic_train_time,
        "Prediction Time": metadata_logistic_prediction_time
    },
    {
        "Model": "Linear SVM",
        "Accuracy": metadata_svm_accuracy,
        "Precision": metadata_svm_precision,
        "Recall": metadata_svm_recall,
        "F1-Score": metadata_svm_f1,
        "Training Time": metadata_svm_train_time,
        "Prediction Time": metadata_svm_prediction_time
    },
    {
        "Model": "Random Forest",
        "Accuracy": metadata_rf_accuracy,
        "Precision": metadata_rf_precision,
        "Recall": metadata_rf_recall,
        "F1-Score": metadata_rf_f1,
        "Training Time": metadata_rf_train_time,
        "Prediction Time": metadata_rf_prediction_time
    }
])

print("\n" + "=" * 80)
print("EXPERIMENT 2 RESULTS")
print("=" * 80)

display(
    metadata_results.round(4)
)

EXPERIMENT 2: TF-IDF + CASE METADATA

--------------------------------------------------------------------------------
1. LOGISTIC REGRESSION
--------------------------------------------------------------------------------
Training Time   : 39.79 seconds
Prediction Time : 0.0276 seconds
Accuracy        : 0.7549
Precision       : 0.7634
Recall          : 0.6905
F1-Score        : 0.7251

--------------------------------------------------------------------------------
2. LINEAR SVM
--------------------------------------------------------------------------------
Training Time   : 8.36 seconds
Prediction Time : 0.0105 seconds
Accuracy        : 0.7886
Precision       : 0.7827
Recall          : 0.7591
F1-Score        : 0.7707

--------------------------------------------------------------------------------
3. RANDOM FOREST
--------------------------------------------------------------------------------
Training Time   : 147.43 seconds
Prediction Time : 1.5312 seconds
Accuracy        : 0.8300


,Model,Accuracy,Precision,Recall,F1-Score,Training Time,Prediction Time
0,Logistic Regression,0.7549,0.7634,0.6905,0.7251,39.7909,0.0276
1,Linear SVM,0.7886,0.7827,0.7591,0.7707,8.3604,0.0105
2,Random Forest,0.8300,0.8523,0.7703,0.8092,147.4326,1.5312


In [ ]:
# ============================================================
# EXPERIMENT 3
# STEP 11: Create Constitutional Article Reference Feature
# ============================================================

import re

print("=" * 80)
print("CREATING CONSTITUTIONAL ARTICLE REFERENCE FEATURE")
print("=" * 80)


def has_constitutional_article(text):
    """
    Detect whether a judgment contains a likely constitutional
    Article reference.

    Examples detected:
        Article 226
        Article 31C
        Article 300A
        Article 300 a
        Article 239 aa
        Articles 14 and 19
        Art. 133

    Ordinary uses such as:
        articles stolen
        articles of material
    are not treated as constitutional references.
    """

    if pd.isna(text):
        return 0

    text = str(text).lower()

    # --------------------------------------------------------
    # Constitutional context keywords
    # --------------------------------------------------------

    constitutional_context = (
        r"(?:constitution|constitutional|fundamental rights|"
        r"directive principles|constitution of india)"
    )

    # --------------------------------------------------------
    # Article / Articles followed by a number
    # --------------------------------------------------------

    pattern = re.compile(
        r"\barticles?\s*[-.]?\s*"
        r"\d{1,3}"
        r"(?:\s*[a-z]{1,2})?"
        r".{0,100}?"
        + constitutional_context,
        flags=re.IGNORECASE
    )

    if pattern.search(text):
        return 1

    # --------------------------------------------------------
    # Handle cases where constitutional context appears
    # immediately before Article reference.
    # --------------------------------------------------------

    reverse_pattern = re.compile(
        constitutional_context +
        r".{0,100}?"
        r"\barticles?\s*[-.]?\s*"
        r"\d{1,3}"
        r"(?:\s*[a-z]{1,2})?",
        flags=re.IGNORECASE
    )

    if reverse_pattern.search(text):
        return 1

    # --------------------------------------------------------
    # Specific "under Article X of the Constitution" structure
    # --------------------------------------------------------

    direct_pattern = re.compile(
        r"\b(?:article|articles)\s*[-.]?\s*"
        r"\d{1,3}"
        r"(?:\s*[a-z]{1,2})?"
        r"(?:\s*(?:and|,)\s*\d{1,3}"
        r"(?:\s*[a-z]{1,2})?)*"
        r"\s+(?:of|under)\s+"
        r"(?:the\s+)?constitution",
        flags=re.IGNORECASE
    )

    if direct_pattern.search(text):
        return 1

    return 0


# ------------------------------------------------------------
# Apply to ORIGINAL judgment text
# ------------------------------------------------------------
# Important:
# We use the original `judgement` column rather than
# `cleaned_judgement`, because punctuation/context can help
# distinguish legal Article references.
# ------------------------------------------------------------

modeling_df["has_article_reference"] = (
    modeling_df["judgement"]
    .apply(has_constitutional_article)
)


# ------------------------------------------------------------
# Inspect distribution
# ------------------------------------------------------------

print("\nFeature created successfully.")

print("\nHas Article Reference:")
print(
    modeling_df["has_article_reference"]
    .value_counts()
    .sort_index()
)

article_count = (
    modeling_df["has_article_reference"]
    .sum()
)

total_count = len(modeling_df)

percentage = (
    article_count / total_count
) * 100

print("\n" + "=" * 80)
print("ARTICLE REFERENCE FEATURE SUMMARY")
print("=" * 80)

print(f"Total Judgments       : {total_count:,}")
print(f"With Article Reference: {article_count:,}")
print(f"Without Reference     : {total_count - article_count:,}")
print(f"Percentage            : {percentage:.2f}%")

CREATING CONSTITUTIONAL ARTICLE REFERENCE FEATURE

Feature created successfully.

Has Article Reference:
has_article_reference
0    32060
1    10109
Name: count, dtype: int64

ARTICLE REFERENCE FEATURE SUMMARY
Total Judgments       : 42,169
With Article Reference: 10,109
Without Reference     : 32,060
Percentage            : 23.97%


In [ ]:
# ============================================================
# EXPERIMENT 3
# STEP 12: ROBUST CONSTITUTIONAL ARTICLE DETECTOR
# ============================================================

import re

print("=" * 80)
print("IMPROVING CONSTITUTIONAL ARTICLE DETECTION")
print("=" * 80)


def has_constitutional_article_v2(text):

    if pd.isna(text):
        return 0

    text = str(text).lower()

    # --------------------------------------------------------
    # Direct and strong constitutional Article patterns
    # --------------------------------------------------------

    direct_patterns = [

        # Article 226 of the Constitution
        r"\barticles?\s*[-.]?\s*\d{1,3}"
        r"(?:\s*[a-z]{1,2})?"
        r"(?:\s*(?:and|,)\s*\d{1,3}"
        r"(?:\s*[a-z]{1,2})?)*"
        r"\s+(?:of|under)\s+"
        r"(?:the\s+)?constitution\b",

        # Article 226 ... Constitution
        r"\barticles?\s*[-.]?\s*\d{1,3}"
        r"(?:\s*[a-z]{1,2})?"
        r".{0,250}?"
        r"\bconstitution\b",

        # Constitution ... Article 226
        r"\bconstitution\b"
        r".{0,250}?"
        r"\barticles?\s*[-.]?\s*\d{1,3}"
        r"(?:\s*[a-z]{1,2})?"
    ]

    for pattern in direct_patterns:

        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE | re.DOTALL
        ):
            return 1

    # --------------------------------------------------------
    # "Art." abbreviation
    # --------------------------------------------------------

    art_pattern = (
        r"\bart\.\s*"
        r"\d{1,3}"
        r"(?:\s*[a-z]{1,2})?"
        r".{0,250}?"
        r"\bconstitution\b"
    )

    if re.search(
        art_pattern,
        text,
        flags=re.IGNORECASE | re.DOTALL
    ):
        return 1

    return 0


# ------------------------------------------------------------
# Apply improved detector
# ------------------------------------------------------------

modeling_df["has_article_reference"] = (
    modeling_df["judgement"]
    .apply(has_constitutional_article_v2)
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

article_present = (
    modeling_df["has_article_reference"].sum()
)

total = len(modeling_df)

print("\n" + "=" * 80)
print("UPDATED ARTICLE REFERENCE SUMMARY")
print("=" * 80)

print(f"Total Judgments       : {total:,}")
print(f"With Article Reference: {article_present:,}")
print(
    f"Without Reference     : "
    f"{total - article_present:,}"
)
print(
    f"Percentage            : "
    f"{article_present / total * 100:.2f}%"
)

print("\nDistribution:")
print(
    modeling_df["has_article_reference"]
    .value_counts()
    .sort_index()
)

IMPROVING CONSTITUTIONAL ARTICLE DETECTION

UPDATED ARTICLE REFERENCE SUMMARY
Total Judgments       : 42,169
With Article Reference: 11,602
Without Reference     : 30,567
Percentage            : 27.51%

Distribution:
has_article_reference
0    30567
1    11602
Name: count, dtype: int64


In [ ]:
# ============================================================
# STEP 13: AUDIT ARTICLE REFERENCE FEATURE
# ============================================================

print("=" * 80)
print("ARTICLE REFERENCE FEATURE AUDIT")
print("=" * 80)

positive_cases = modeling_df[
    modeling_df["has_article_reference"] == 1
].sample(
    n=5,
    random_state=42
)

negative_cases = modeling_df[
    modeling_df["has_article_reference"] == 0
].sample(
    n=5,
    random_state=42
)

print("\n" + "-" * 80)
print("POSITIVE CASES")
print("-" * 80)

for i, (_, row) in enumerate(positive_cases.iterrows(), 1):

    print(f"\nPositive Case {i}")
    print("-" * 40)

    text = row["judgement"]

    # Show first 1500 characters
    print(text[:1500])


print("\n\n" + "-" * 80)
print("NEGATIVE CASES")
print("-" * 80)

for i, (_, row) in enumerate(negative_cases.iterrows(), 1):

    print(f"\nNegative Case {i}")
    print("-" * 40)

    text = row["judgement"]

    print(text[:1500])

ARTICLE REFERENCE FEATURE AUDIT

--------------------------------------------------------------------------------
POSITIVE CASES
--------------------------------------------------------------------------------

Positive Case 1
----------------------------------------
cessary implication have undoubtedly retrospective effect.
632d-f harbhajan singh v. state of punjab 1977 2 s.l.r.
180 ex.
major.
n.c singhal v. director general armed forces medical service a.i.r. 1972 s.c. 628 state of mysore v. m.n. krishna murty ors., 1973 2 s.c.r.
575 raj kumar v. union of india ors., 1975 3 s.c.r. 963 wing commander j. kumar v. union of india ors.
1982 2 s.c.c.
116 b.s. vadera v. union of india ors., 1968 3 s.c.r.
575 discussed.
13.
the haryana government cannot take away the accrued rights of the petitioners and the appellants by making amendment of the rules with retrospective effect.
the impugned rule 4  ii  of the punjab government national emergency  concessions  rule, 1965, as amended by the ha

In [ ]:
# ============================================================
# EXPERIMENT 3
# STEP 14: ADD ARTICLE REFERENCE FEATURE
# ============================================================

from scipy.sparse import hstack
from scipy.sparse import csr_matrix

print("=" * 80)
print("ADDING ARTICLE REFERENCE FEATURE")
print("=" * 80)

# ------------------------------------------------------------
# IMPORTANT:
# Use the already-created train/test split.
# Do NOT split the dataset again.
# ------------------------------------------------------------

train_article_feature = (
    train_df["judgement"]
    .apply(has_constitutional_article_v2)
    .values
    .reshape(-1, 1)
)

test_article_feature = (
    test_df["judgement"]
    .apply(has_constitutional_article_v2)
    .values
    .reshape(-1, 1)
)

# Convert to sparse matrices
X_train_article = csr_matrix(
    train_article_feature
)

X_test_article = csr_matrix(
    test_article_feature
)

print("Article feature matrices created.")

print("\nTraining Article Feature Shape:")
print(X_train_article.shape)

print("\nTesting Article Feature Shape:")
print(X_test_article.shape)


# ------------------------------------------------------------
# Combine:
#
# TF-IDF
# + Metadata
# + Article Reference
# ------------------------------------------------------------

X_train_final = hstack(
    [
        X_train_metadata_combined,
        X_train_article
    ],
    format="csr"
)

X_test_final = hstack(
    [
        X_test_metadata_combined,
        X_test_article
    ],
    format="csr"
)

print("\n" + "=" * 80)
print("FINAL FEATURE MATRICES")
print("=" * 80)

print("\nTraining Matrix:")
print(X_train_final.shape)

print("\nTesting Matrix:")
print(X_test_final.shape)

print("\nTotal Features:")
print(X_train_final.shape[1])

ADDING ARTICLE REFERENCE FEATURE
Article feature matrices created.

Training Article Feature Shape:
(33735, 1)

Testing Article Feature Shape:
(8434, 1)

FINAL FEATURE MATRICES

Training Matrix:
(33735, 50015)

Testing Matrix:
(8434, 50015)

Total Features:
50015


In [ ]:
# ============================================================
# EXPERIMENT 3
# MODEL TRAINING
# TF-IDF + CASE METADATA + ARTICLE REFERENCE
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import time
import pandas as pd

print("=" * 80)
print("EXPERIMENT 3: TF-IDF + CASE METADATA + ARTICLE REFERENCE")
print("=" * 80)

experiment3_results = []


# ============================================================
# 1. LOGISTIC REGRESSION
# ============================================================

print("\n" + "-" * 80)
print("1. LOGISTIC REGRESSION")
print("-" * 80)

model3_logistic = LogisticRegression(
    max_iter=1000,
    random_state=42
)

start_time = time.time()

model3_logistic.fit(
    X_train_final,
    y_train
)

train_time = time.time() - start_time

start_time = time.time()

pred_logistic = model3_logistic.predict(
    X_test_final
)

prediction_time = time.time() - start_time

accuracy = accuracy_score(y_test, pred_logistic)
precision = precision_score(y_test, pred_logistic)
recall = recall_score(y_test, pred_logistic)
f1 = f1_score(y_test, pred_logistic)

print(f"Training Time   : {train_time:.2f} seconds")
print(f"Prediction Time : {prediction_time:.4f} seconds")
print(f"Accuracy        : {accuracy:.4f}")
print(f"Precision       : {precision:.4f}")
print(f"Recall          : {recall:.4f}")
print(f"F1-Score        : {f1:.4f}")

experiment3_results.append({
    "Model": "Logistic Regression",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1,
    "Training Time": train_time,
    "Prediction Time": prediction_time
})


# ============================================================
# 2. LINEAR SVM
# ============================================================

print("\n" + "-" * 80)
print("2. LINEAR SVM")
print("-" * 80)

model3_svm = LinearSVC(
    random_state=42
)

start_time = time.time()

model3_svm.fit(
    X_train_final,
    y_train
)

train_time = time.time() - start_time

start_time = time.time()

pred_svm = model3_svm.predict(
    X_test_final
)

prediction_time = time.time() - start_time

accuracy = accuracy_score(y_test, pred_svm)
precision = precision_score(y_test, pred_svm)
recall = recall_score(y_test, pred_svm)
f1 = f1_score(y_test, pred_svm)

print(f"Training Time   : {train_time:.2f} seconds")
print(f"Prediction Time : {prediction_time:.4f} seconds")
print(f"Accuracy        : {accuracy:.4f}")
print(f"Precision       : {precision:.4f}")
print(f"Recall          : {recall:.4f}")
print(f"F1-Score        : {f1:.4f}")

experiment3_results.append({
    "Model": "Linear SVM",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1,
    "Training Time": train_time,
    "Prediction Time": prediction_time
})


# ============================================================
# 3. RANDOM FOREST
# ============================================================

print("\n" + "-" * 80)
print("3. RANDOM FOREST")
print("-" * 80)

model3_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

model3_rf.fit(
    X_train_final,
    y_train
)

train_time = time.time() - start_time

start_time = time.time()

pred_rf = model3_rf.predict(
    X_test_final
)

prediction_time = time.time() - start_time

accuracy = accuracy_score(y_test, pred_rf)
precision = precision_score(y_test, pred_rf)
recall = recall_score(y_test, pred_rf)
f1 = f1_score(y_test, pred_rf)

print(f"Training Time   : {train_time:.2f} seconds")
print(f"Prediction Time : {prediction_time:.4f} seconds")
print(f"Accuracy        : {accuracy:.4f}")
print(f"Precision       : {precision:.4f}")
print(f"Recall          : {recall:.4f}")
print(f"F1-Score        : {f1:.4f}")

experiment3_results.append({
    "Model": "Random Forest",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1,
    "Training Time": train_time,
    "Prediction Time": prediction_time
})


# ============================================================
# RESULTS TABLE
# ============================================================

experiment3_results_df = pd.DataFrame(
    experiment3_results
)

print("\n" + "=" * 80)
print("EXPERIMENT 3 RESULTS")
print("=" * 80)

display(
    experiment3_results_df.round(4)
)

EXPERIMENT 3: TF-IDF + CASE METADATA + ARTICLE REFERENCE

--------------------------------------------------------------------------------
1. LOGISTIC REGRESSION
--------------------------------------------------------------------------------
Training Time   : 28.43 seconds
Prediction Time : 0.0132 seconds
Accuracy        : 0.7553
Precision       : 0.7627
Recall          : 0.6928
F1-Score        : 0.7260

--------------------------------------------------------------------------------
2. LINEAR SVM
--------------------------------------------------------------------------------
Training Time   : 8.62 seconds
Prediction Time : 0.0143 seconds
Accuracy        : 0.7882
Precision       : 0.7821
Recall          : 0.7591
F1-Score        : 0.7704

--------------------------------------------------------------------------------
3. RANDOM FOREST
--------------------------------------------------------------------------------
Training Time   : 149.45 seconds
Prediction Time : 1.4990 seconds
Accur

,Model,Accuracy,Precision,Recall,F1-Score,Training Time,Prediction Time
0,Logistic Regression,0.7553,0.7627,0.6928,0.7260,28.4285,0.0132
1,Linear SVM,0.7882,0.7821,0.7591,0.7704,8.6245,0.0143
2,Random Forest,0.8270,0.8475,0.7687,0.8062,149.4515,1.4990


In [ ]:
# ============================================================
# RANDOM FOREST HYPERPARAMETER TUNING
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import pandas as pd
import time

print("=" * 80)
print("RANDOM FOREST HYPERPARAMETER TUNING")
print("=" * 80)

# ------------------------------------------------------------
# Candidate configurations
# ------------------------------------------------------------

rf_configs = [

    {
        "name": "RF-1",
        "n_estimators": 150,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },

    {
        "name": "RF-2",
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },

    {
        "name": "RF-3",
        "n_estimators": 150,
        "max_depth": 30,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },

    {
        "name": "RF-4",
        "n_estimators": 150,
        "max_depth": 20,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },

    {
        "name": "RF-5",
        "n_estimators": 150,
        "max_depth": None,
        "min_samples_split": 5,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },

    {
        "name": "RF-6",
        "n_estimators": 150,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": "sqrt"
    }
]


# ------------------------------------------------------------
# Store results
# ------------------------------------------------------------

rf_tuning_results = []


# ------------------------------------------------------------
# Train each configuration
# ------------------------------------------------------------

for config in rf_configs:

    print("\n" + "-" * 80)
    print(config["name"])
    print("-" * 80)

    print(
        f"n_estimators      : {config['n_estimators']}"
    )
    print(
        f"max_depth         : {config['max_depth']}"
    )
    print(
        f"min_samples_split : {config['min_samples_split']}"
    )
    print(
        f"min_samples_leaf  : {config['min_samples_leaf']}"
    )
    print(
        f"max_features      : {config['max_features']}"
    )

    model = RandomForestClassifier(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        min_samples_split=config["min_samples_split"],
        min_samples_leaf=config["min_samples_leaf"],
        max_features=config["max_features"],
        random_state=42,
        n_jobs=-1
    )

    start_time = time.time()

    model.fit(
        X_train_metadata_combined,
        y_train
    )

    training_time = time.time() - start_time

    start_time = time.time()

    predictions = model.predict(
        X_test_metadata_combined
    )

    prediction_time = time.time() - start_time

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions
    )

    recall = recall_score(
        y_test,
        predictions
    )

    f1 = f1_score(
        y_test,
        predictions
    )

    print(f"Training Time   : {training_time:.2f} seconds")
    print(f"Prediction Time : {prediction_time:.4f} seconds")
    print(f"Accuracy        : {accuracy:.4f}")
    print(f"Precision       : {precision:.4f}")
    print(f"Recall          : {recall:.4f}")
    print(f"F1-Score        : {f1:.4f}")

    rf_tuning_results.append({

        "Configuration": config["name"],

        "n_estimators":
            config["n_estimators"],

        "max_depth":
            config["max_depth"],

        "min_samples_split":
            config["min_samples_split"],

        "min_samples_leaf":
            config["min_samples_leaf"],

        "max_features":
            config["max_features"],

        "Accuracy":
            accuracy,

        "Precision":
            precision,

        "Recall":
            recall,

        "F1-Score":
            f1,

        "Training Time":
            training_time,

        "Prediction Time":
            prediction_time
    })


# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------

rf_tuning_results_df = pd.DataFrame(
    rf_tuning_results
)

rf_tuning_results_df = (
    rf_tuning_results_df
    .sort_values(
        by="F1-Score",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 80)
print("RANDOM FOREST TUNING RESULTS")
print("=" * 80)

display(
    rf_tuning_results_df.round(4)
)

RANDOM FOREST HYPERPARAMETER TUNING

--------------------------------------------------------------------------------
RF-1
--------------------------------------------------------------------------------
n_estimators      : 150
max_depth         : None
min_samples_split : 2
min_samples_leaf  : 1
max_features      : sqrt
Training Time   : 228.44 seconds
Prediction Time : 3.0613 seconds
Accuracy        : 0.8387
Precision       : 0.8617
Recall          : 0.7809
F1-Score        : 0.8193

--------------------------------------------------------------------------------
RF-2
--------------------------------------------------------------------------------
n_estimators      : 200
max_depth         : None
min_samples_split : 2
min_samples_leaf  : 1
max_features      : sqrt
Training Time   : 292.65 seconds
Prediction Time : 3.8625 seconds
Accuracy        : 0.8411
Precision       : 0.8620
Recall          : 0.7865
F1-Score        : 0.8225

-----------------------------------------------------------

,Configuration,n_estimators,max_depth,min_samples_split,min_samples_leaf,max_features,Accuracy,Precision,Recall,F1-Score,Training Time,Prediction Time
0,RF-6,150,NaN,2,2,sqrt,0.8401,0.8530,0.7953,0.8232,192.9647,2.2071
1,RF-2,200,NaN,2,1,sqrt,0.8411,0.8620,0.7865,0.8225,292.6533,3.8625
2,RF-1,150,NaN,2,1,sqrt,0.8387,0.8617,0.7809,0.8193,228.4398,3.0613
3,RF-5,150,NaN,5,1,sqrt,0.8366,0.8511,0.7890,0.8189,208.3703,2.3615
4,RF-3,150,30.0,2,1,sqrt,0.8304,0.8591,0.7629,0.8082,112.8997,2.0902
5,RF-4,150,20.0,2,1,sqrt,0.8154,0.8696,0.7125,0.7832,57.5386,2.8778


In [ ]:
# ============================================================
# STEP 17: INSPECT CURRENT NOTEBOOK STATE
# ============================================================

print("=" * 80)
print("CHECKING AVAILABLE MODELING OBJECTS")
print("=" * 80)

objects_to_check = [
    "modeling_df",
    "train_df",
    "test_df",

    "X_train_tfidf",
    "X_test_tfidf",

    "X_train_metadata_combined",
    "X_test_metadata_combined",

    "X_train_final",
    "X_test_final",

    "y_train",
    "y_test",

    "logistic_model",
    "svm_model",
    "rf_model",

    "model3_logistic",
    "model3_svm",
    "model3_rf",

    "metadata_logistic",
    "metadata_svm",
    "metadata_rf",

    "rf_tuning_results_df"
]

for obj in objects_to_check:

    if obj in globals():
        print(f"✓ {obj}")
    else:
        print(f"✗ {obj}")

print("\n" + "=" * 80)
print("CHECKING DATASET COLUMNS")
print("=" * 80)

if "modeling_df" in globals():

    print(modeling_df.columns.tolist())
    print(f"\nShape: {modeling_df.shape}")

elif "train_df" in globals():

    print(train_df.columns.tolist())
    print(f"\nTrain shape: {train_df.shape}")

else:

    print("Modeling dataframe is not currently available.")

CHECKING AVAILABLE MODELING OBJECTS
✓ modeling_df
✓ train_df
✓ test_df
✗ X_train_tfidf
✗ X_test_tfidf
✓ X_train_metadata_combined
✓ X_test_metadata_combined
✓ X_train_final
✓ X_test_final
✓ y_train
✓ y_test
✗ logistic_model
✗ svm_model
✗ rf_model
✓ model3_logistic
✓ model3_svm
✓ model3_rf
✓ metadata_logistic
✓ metadata_svm
✓ metadata_rf
✓ rf_tuning_results_df

CHECKING DATASET COLUMNS
['case_category', 'case_type', 'case_info', 'judgement', 'label', 'char_count', 'word_count', 'sentence_count', 'cleaned_judgement', 'has_article_reference']

Shape: (42169, 10)


In [ ]:
# ============================================================
# STEP 18: FIND EXISTING TF-IDF / VECTORIZER OBJECTS
# ============================================================

print("=" * 80)
print("SEARCHING FOR TF-IDF / VECTORIZER OBJECTS")
print("=" * 80)

found_objects = []

# Take a snapshot before iterating
current_objects = list(globals().items())

for name, obj in current_objects:

    name_lower = name.lower()

    if (
        "tfidf" in name_lower
        or "vectorizer" in name_lower
    ):

        found_objects.append(name)

        print(f"\n✓ {name}")
        print(f"  Type: {type(obj)}")

        # Inspect vectorizer parameters if available
        if hasattr(obj, "get_params"):

            try:
                params = obj.get_params()

                print("  Parameters:")

                for key in [
                    "ngram_range",
                    "max_features",
                    "min_df",
                    "max_df",
                    "sublinear_tf",
                    "lowercase",
                    "stop_words"
                ]:

                    if key in params:
                        print(
                            f"    {key}: {params[key]}"
                        )

            except Exception:
                pass


print("\n" + "=" * 80)

if found_objects:

    print(
        f"Found {len(found_objects)} "
        "TF-IDF/vectorizer-related object(s)."
    )

else:

    print(
        "No TF-IDF/vectorizer objects found."
    )

SEARCHING FOR TF-IDF / VECTORIZER OBJECTS

✓ TfidfVectorizer
  Type: <class 'type'>

✓ tfidf_vectorizer
  Type: <class 'sklearn.feature_extraction.text.TfidfVectorizer'>
  Parameters:
    ngram_range: (1, 1)
    max_features: 50000
    min_df: 1
    max_df: 1.0
    sublinear_tf: False
    lowercase: True
    stop_words: None

Found 2 TF-IDF/vectorizer-related object(s).


In [ ]:
# ============================================================
# STEP 19: INSPECT EXISTING RANDOM FOREST MODELS
# ============================================================

print("=" * 80)
print("RANDOM FOREST MODEL CONFIGURATION")
print("=" * 80)


models_to_check = [
    ("model3_rf", model3_rf),
    ("metadata_rf", metadata_rf)
]


for model_name, model in models_to_check:

    print("\n" + "-" * 80)
    print(model_name)
    print("-" * 80)

    params = model.get_params()

    important_params = [
        "n_estimators",
        "criterion",
        "max_depth",
        "min_samples_split",
        "min_samples_leaf",
        "max_features",
        "bootstrap",
        "class_weight",
        "random_state",
        "n_jobs"
    ]

    for param in important_params:

        print(
            f"{param:20s}: "
            f"{params.get(param)}"
        )


    print(
        f"\nNumber of trees: "
        f"{len(model.estimators_)}"
    )

    print(
        f"Number of features seen during fit: "
        f"{model.n_features_in_}"
    )

RANDOM FOREST MODEL CONFIGURATION

--------------------------------------------------------------------------------
model3_rf
--------------------------------------------------------------------------------
n_estimators        : 100
criterion           : gini
max_depth           : None
min_samples_split   : 2
min_samples_leaf    : 1
max_features        : sqrt
bootstrap           : True
class_weight        : None
random_state        : 42
n_jobs              : -1

Number of trees: 100
Number of features seen during fit: 50015

--------------------------------------------------------------------------------
metadata_rf
--------------------------------------------------------------------------------
n_estimators        : 100
criterion           : gini
max_depth           : None
min_samples_split   : 2
min_samples_leaf    : 1
max_features        : sqrt
bootstrap           : True
class_weight        : None
random_state        : 42
n_jobs              : -1

Number of trees: 100
Number of feat

In [ ]:
# ============================================================
# STEP 20: INSPECT TF-IDF VECTOR REPRESENTATION
# ============================================================

print("=" * 80)
print("TF-IDF VECTOR REPRESENTATION INSPECTION")
print("=" * 80)

print("\nVectorizer vocabulary size:")
print(len(tfidf_vectorizer.vocabulary_))

print("\nStored feature names:")
feature_names = tfidf_vectorizer.get_feature_names_out()

print("Number of features:", len(feature_names))
print("First 30 features:")
print(feature_names[:30])

print("\n" + "-" * 80)
print("CURRENT TRAINING DATA")
print("-" * 80)

print("Training samples:", len(train_df))

print("\nTraining columns:")
print(train_df.columns.tolist())

print("\nFirst training judgment:")
print(train_df["cleaned_judgement"].iloc[0][:1000])

print("\n" + "-" * 80)
print("TF-IDF TRANSFORMATION CHECK")
print("-" * 80)

sample_text = train_df["cleaned_judgement"].iloc[:5]

sample_matrix = tfidf_vectorizer.transform(sample_text)

print("Sample matrix shape:")
print(sample_matrix.shape)

print("Non-zero values in first sample:")
print(sample_matrix[0].nnz)

print("\nTF-IDF inspection completed.")

TF-IDF VECTOR REPRESENTATION INSPECTION

Vectorizer vocabulary size:
50000

Stored feature names:
Number of features: 50000
First 30 features:
['000' '0000' '00000' '000000' '0001' '0007' '001' '002' '003' '0030'
 '004' '0045' '005' '006' '007' '008' '009' '00a' '00p' '00pm' '010'
 '0100' '011' '012' '013' '014' '015' '016' '017' '018']

--------------------------------------------------------------------------------
CURRENT TRAINING DATA
--------------------------------------------------------------------------------
Training samples: 33735

Training columns:
['case_category', 'case_type', 'case_info', 'judgement', 'label', 'char_count', 'word_count', 'sentence_count', 'cleaned_judgement']

First training judgment:
partition document containing list property allotted party recital past event registration whether necessary whether admissible evidence prove factum partition headnote party decendants common ancestor two son two branch family joint property agricultural residential agricu

In [ ]:
# ============================================================
# STEP 21: CHECK POTENTIAL OUTCOME-LANGUAGE LEAKAGE
# ============================================================

import re
import pandas as pd

print("=" * 80)
print("CHECKING POTENTIAL OUTCOME-LANGUAGE LEAKAGE")
print("=" * 80)


outcome_patterns = {

    "dismissed": r"\bdismissed\b",
    "rejected": r"\brejected\b",
    "denied": r"\bdenied\b",

    "allowed": r"\ballowed\b",
    "granted": r"\bgranted\b",

    "set_aside": r"\bset aside\b",
    "quashed": r"\bquashed\b",

    "disposed": r"\bdisposed\b",
    "appeal_allowed": r"\bappeal[s]?\s+(is|are)?\s*allowed\b",
    "appeal_dismissed": r"\bappeal[s]?\s+(is|are)?\s*dismissed\b"
}


def count_outcome_terms(df):

    results = []

    for name, pattern in outcome_patterns.items():

        matches = df["cleaned_judgement"].str.contains(
            pattern,
            regex=True,
            case=False,
            na=False
        )

        results.append({
            "Term": name,
            "Count": matches.sum(),
            "Percentage": matches.mean() * 100
        })

    return pd.DataFrame(results)


print("\n" + "-" * 80)
print("FULL DATASET")
print("-" * 80)

full_outcome_stats = count_outcome_terms(
    modeling_df
)

display(
    full_outcome_stats.round(2)
)


print("\n" + "-" * 80)
print("TRAINING DATA")
print("-" * 80)

train_outcome_stats = count_outcome_terms(
    train_df
)

display(
    train_outcome_stats.round(2)
)


print("\n" + "-" * 80)
print("TESTING DATA")
print("-" * 80)

test_outcome_stats = count_outcome_terms(
    test_df
)

display(
    test_outcome_stats.round(2)
)


# ------------------------------------------------------------
# Label-wise analysis
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("OUTCOME WORDS BY LABEL")
print("=" * 80)

for term_name, pattern in outcome_patterns.items():

    matches = modeling_df[
        "cleaned_judgement"
    ].str.contains(
        pattern,
        regex=True,
        case=False,
        na=False
    )

    if matches.sum() == 0:
        continue

    label_distribution = (
        modeling_df.loc[matches, "label"]
        .value_counts()
        .sort_index()
    )

    print(
        f"\n{term_name}:"
    )

    print(
        label_distribution.to_dict()
    )

CHECKING POTENTIAL OUTCOME-LANGUAGE LEAKAGE

--------------------------------------------------------------------------------
FULL DATASET
--------------------------------------------------------------------------------


/tmp/ipykernel_1019/1457204863.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matches = df["cleaned_judgement"].str.contains(


,Term,Count,Percentage
0,dismissed,27656,65.58
1,rejected,13318,31.58
2,denied,7678,18.21
3,allowed,27240,64.60
4,granted,19531,46.32
5,set_aside,21562,51.13
6,quashed,4931,11.69
7,disposed,11454,27.16
8,appeal_allowed,11258,26.70
9,appeal_dismissed,8880,21.06



--------------------------------------------------------------------------------
TRAINING DATA
--------------------------------------------------------------------------------


/tmp/ipykernel_1019/1457204863.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matches = df["cleaned_judgement"].str.contains(


,Term,Count,Percentage
0,dismissed,22156,65.68
1,rejected,10666,31.62
2,denied,6151,18.23
3,allowed,21802,64.63
4,granted,15659,46.42
5,set_aside,17254,51.15
6,quashed,3964,11.75
7,disposed,9166,27.17
8,appeal_allowed,9000,26.68
9,appeal_dismissed,7123,21.11


... (2 outputs left out)

,Term,Count,Percentage
0,dismissed,5500,65.21
1,rejected,2652,31.44
2,denied,1527,18.11
3,allowed,5438,64.48
4,granted,3872,45.91
5,set_aside,4308,51.08
6,quashed,967,11.47
7,disposed,2288,27.13
8,appeal_allowed,2258,26.77
9,appeal_dismissed,1757,20.83



OUTCOME WORDS BY LABEL

dismissed:
{0: 17970, 1: 9686}

rejected:
{0: 7315, 1: 6003}

denied:
{0: 4049, 1: 3629}

allowed:
{0: 10144, 1: 17096}

granted:
{0: 10003, 1: 9528}

set_aside:
{0: 7207, 1: 14355}

quashed:
{0: 2084, 1: 2847}

disposed:
{0: 7342, 1: 4112}


/tmp/ipykernel_1019/1457204863.py:104: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ].str.contains(



appeal_allowed:
{0: 1083, 1: 10175}

appeal_dismissed:
{0: 7740, 1: 1140}


In [ ]:
# ============================================================
# REPRODUCTION OF ORIGINAL EXPERIMENT 1
# STEP 1: Original TF-IDF Configuration
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

# Original TF-IDF configuration from the trained notebook
tfidf_original = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95,
    sublinear_tf=True
)

print("Fitting original TF-IDF on training data...")

# Fit ONLY on training data
X_train_tfidf_original = tfidf_original.fit_transform(X_train)

# Transform test data using the same fitted vectorizer
X_test_tfidf_original = tfidf_original.transform(X_test)

print("\nOriginal TF-IDF reproduced successfully!")
print("Training Feature Matrix Shape :", X_train_tfidf_original.shape)
print("Testing Feature Matrix Shape  :", X_test_tfidf_original.shape)
print("Vocabulary Size               :", len(tfidf_original.vocabulary_))

Fitting original TF-IDF on training data...


NameError: name 'X_train' is not defined

In [ ]:
# ============================================================
# REPRODUCTION OF ORIGINAL EXPERIMENT 1
# STEP 1A: Restore X_train and X_test
# ============================================================

X_train = train_df["cleaned_judgement"]
X_test = test_df["cleaned_judgement"]

print("=" * 80)
print("ORIGINAL TEXT FEATURES RESTORED")
print("=" * 80)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("\nFirst training document:")
print(X_train.iloc[0][:500])

ORIGINAL TEXT FEATURES RESTORED
X_train shape: (33735,)
X_test shape : (8434,)

First training document:
partition document containing list property allotted party recital past event registration whether necessary whether admissible evidence prove factum partition headnote party decendants common ancestor two son two branch family joint property agricultural residential agricultural land partitioned 1955 name respective party duly mutated revenue record followed partition residential property including house ghers ghetwars etc factum partition embodied memorandum partition exh dated august 1955 bea


In [ ]:
# ============================================================
# REPRODUCTION OF ORIGINAL EXPERIMENT 1
# STEP 1B: Original TF-IDF Configuration
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_original = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95,
    sublinear_tf=True
)

print("=" * 80)
print("FITTING ORIGINAL TF-IDF")
print("=" * 80)

X_train_tfidf_original = tfidf_original.fit_transform(X_train)
X_test_tfidf_original = tfidf_original.transform(X_test)

print("\nOriginal TF-IDF reproduced successfully!")

print("Training Feature Matrix :", X_train_tfidf_original.shape)
print("Testing Feature Matrix  :", X_test_tfidf_original.shape)
print("Vocabulary Size         :", len(tfidf_original.vocabulary_))

FITTING ORIGINAL TF-IDF

Original TF-IDF reproduced successfully!
Training Feature Matrix : (33735, 50000)
Testing Feature Matrix  : (8434, 50000)
Vocabulary Size         : 50000


In [ ]:
# ============================================================
# REPRODUCTION OF ORIGINAL EXPERIMENT 1
# STEP 2: Original Random Forest
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
import time

print("=" * 80)
print("REPRODUCING ORIGINAL RANDOM FOREST")
print("=" * 80)

rf_original = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

train_start = time.time()

rf_original.fit(
    X_train_tfidf_original,
    y_train
)

train_time = time.time() - train_start

# ------------------------------------------------------------
# Prediction
# ------------------------------------------------------------

predict_start = time.time()

y_pred_original_rf = rf_original.predict(
    X_test_tfidf_original
)

prediction_time = time.time() - predict_start

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test,
    y_pred_original_rf
)

precision = precision_score(
    y_test,
    y_pred_original_rf
)

recall = recall_score(
    y_test,
    y_pred_original_rf
)

f1 = f1_score(
    y_test,
    y_pred_original_rf
)

print("\n" + "=" * 80)
print("ORIGINAL RANDOM FOREST REPRODUCTION RESULTS")
print("=" * 80)

print(f"\nTraining Time   : {train_time:.2f} seconds")
print(f"Prediction Time : {prediction_time:.4f} seconds")

print(f"\nAccuracy        : {accuracy:.4f}")
print(f"Precision       : {precision:.4f}")
print(f"Recall          : {recall:.4f}")
print(f"F1-Score        : {f1:.4f}")

print("\n" + "-" * 80)
print("CLASSIFICATION REPORT")
print("-" * 80)

print(
    classification_report(
        y_test,
        y_pred_original_rf,
        target_names=["Rejected", "Accepted"]
    )
)

REPRODUCING ORIGINAL RANDOM FOREST

ORIGINAL RANDOM FOREST REPRODUCTION RESULTS

Training Time   : 408.06 seconds
Prediction Time : 5.8287 seconds

Accuracy        : 0.9132
Precision       : 0.9223
Recall          : 0.8896
F1-Score        : 0.9056

--------------------------------------------------------------------------------
CLASSIFICATION REPORT
--------------------------------------------------------------------------------
              precision    recall  f1-score   support

    Rejected       0.91      0.93      0.92      4486
    Accepted       0.92      0.89      0.91      3948

    accuracy                           0.91      8434
   macro avg       0.91      0.91      0.91      8434
weighted avg       0.91      0.91      0.91      8434



In [ ]:
# ============================================================
# EXPERIMENT 2 — ORIGINAL TF-IDF + CASE METADATA
# STEP 2A: Combine Feature Matrices
# ============================================================

from scipy.sparse import hstack

print("=" * 80)
print("EXPERIMENT 2: ORIGINAL TF-IDF + CASE METADATA")
print("=" * 80)

# Combine original TF-IDF with the already-created metadata features
X_train_exp2 = hstack([
    X_train_tfidf_original,
    X_train_metadata_combined
]).tocsr()

X_test_exp2 = hstack([
    X_test_tfidf_original,
    X_test_metadata_combined
]).tocsr()

print("\nFeature combination completed successfully.")

print("\nOriginal TF-IDF Features:")
print("Training:", X_train_tfidf_original.shape)
print("Testing :", X_test_tfidf_original.shape)

print("\nMetadata Features:")
print("Training:", X_train_metadata_combined.shape)
print("Testing :", X_test_metadata_combined.shape)

print("\nFinal Experiment 2 Features:")
print("Training:", X_train_exp2.shape)
print("Testing :", X_test_exp2.shape)

print("\nTotal Features:", X_train_exp2.shape[1])

EXPERIMENT 2: ORIGINAL TF-IDF + CASE METADATA

Feature combination completed successfully.

Original TF-IDF Features:
Training: (33735, 50000)
Testing : (8434, 50000)

Metadata Features:
Training: (33735, 50014)
Testing : (8434, 50014)

Final Experiment 2 Features:
Training: (33735, 100014)
Testing : (8434, 100014)

Total Features: 100014


In [ ]:
# ============================================================
# CHECK METADATA MATRIX OBJECTS
# ============================================================

print("=" * 80)
print("CHECKING METADATA FEATURE MATRICES")
print("=" * 80)

metadata_objects = [
    "X_train_metadata",
    "X_test_metadata",
    "X_train_metadata_combined",
    "X_test_metadata_combined"
]

for name in metadata_objects:
    if name in globals():
        obj = globals()[name]
        print(f"\n{name}")
        print("Shape:", obj.shape)
    else:
        print(f"\n{name}: NOT AVAILABLE")

CHECKING METADATA FEATURE MATRICES

X_train_metadata
Shape: (33735, 14)

X_test_metadata
Shape: (8434, 14)

X_train_metadata_combined
Shape: (33735, 50014)

X_test_metadata_combined
Shape: (8434, 50014)


In [ ]:
# ============================================================
# EXPERIMENT 2 — ORIGINAL TF-IDF + CASE METADATA
# STEP 2B: Correct Feature Combination
# ============================================================

from scipy.sparse import hstack

print("=" * 80)
print("CREATING CORRECT EXPERIMENT 2 FEATURE MATRICES")
print("=" * 80)

X_train_exp2 = hstack([
    X_train_tfidf_original,
    X_train_metadata
]).tocsr()

X_test_exp2 = hstack([
    X_test_tfidf_original,
    X_test_metadata
]).tocsr()

print("\nFeature combination completed successfully.")

print("\nTraining Matrix:", X_train_exp2.shape)
print("Testing Matrix :", X_test_exp2.shape)
print("Total Features :", X_train_exp2.shape[1])

CREATING CORRECT EXPERIMENT 2 FEATURE MATRICES

Feature combination completed successfully.

Training Matrix: (33735, 50014)
Testing Matrix : (8434, 50014)
Total Features : 50014


In [ ]:
# ============================================================
# EXPERIMENT 2 — TF-IDF + CASE METADATA
# STEP 2C: Train and Evaluate Random Forest
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
import time

print("=" * 80)
print("EXPERIMENT 2: ORIGINAL TF-IDF + CASE METADATA")
print("=" * 80)

rf_exp2 = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# Training
train_start = time.time()

rf_exp2.fit(
    X_train_exp2,
    y_train
)

train_time_exp2 = time.time() - train_start

# Prediction
predict_start = time.time()

y_pred_exp2 = rf_exp2.predict(
    X_test_exp2
)

prediction_time_exp2 = time.time() - predict_start

# Evaluation
accuracy_exp2 = accuracy_score(y_test, y_pred_exp2)
precision_exp2 = precision_score(y_test, y_pred_exp2)
recall_exp2 = recall_score(y_test, y_pred_exp2)
f1_exp2 = f1_score(y_test, y_pred_exp2)

print("\n" + "=" * 80)
print("EXPERIMENT 2 RESULTS")
print("=" * 80)

print(f"\nTraining Time   : {train_time_exp2:.2f} seconds")
print(f"Prediction Time : {prediction_time_exp2:.4f} seconds")

print(f"\nAccuracy        : {accuracy_exp2:.4f}")
print(f"Precision       : {precision_exp2:.4f}")
print(f"Recall          : {recall_exp2:.4f}")
print(f"F1-Score        : {f1_exp2:.4f}")

print("\n" + "-" * 80)
print("CLASSIFICATION REPORT")
print("-" * 80)

print(
    classification_report(
        y_test,
        y_pred_exp2,
        target_names=["Rejected", "Accepted"]
    )
)

EXPERIMENT 2: ORIGINAL TF-IDF + CASE METADATA

EXPERIMENT 2 RESULTS

Training Time   : 404.21 seconds
Prediction Time : 5.4775 seconds

Accuracy        : 0.9140
Precision       : 0.9264
Recall          : 0.8868
F1-Score        : 0.9062

--------------------------------------------------------------------------------
CLASSIFICATION REPORT
--------------------------------------------------------------------------------
              precision    recall  f1-score   support

    Rejected       0.90      0.94      0.92      4486
    Accepted       0.93      0.89      0.91      3948

    accuracy                           0.91      8434
   macro avg       0.92      0.91      0.91      8434
weighted avg       0.91      0.91      0.91      8434



In [ ]:
# ============================================================
# EXPERIMENT 3 — TF-IDF + CASE METADATA + ARTICLE REFERENCE
# STEP 3A: Check Article Feature Matrices
# ============================================================

print("=" * 80)
print("CHECKING ARTICLE FEATURE MATRICES")
print("=" * 80)

article_objects = [
    "X_train_article",
    "X_test_article",
    "X_train_article_feature",
    "X_test_article_feature"
]

for name in article_objects:
    if name in globals():
        obj = globals()[name]
        print(f"\n{name}")
        print("Shape:", obj.shape)
    else:
        print(f"\n{name}: NOT AVAILABLE")

CHECKING ARTICLE FEATURE MATRICES

X_train_article
Shape: (33735, 1)

X_test_article
Shape: (8434, 1)

X_train_article_feature: NOT AVAILABLE

X_test_article_feature: NOT AVAILABLE


In [ ]:
# ============================================================
# EXPERIMENT 3 — TF-IDF + CASE METADATA + ARTICLE REFERENCE
# STEP 3B: Combine All Features
# ============================================================

from scipy.sparse import hstack

print("=" * 80)
print("CREATING EXPERIMENT 3 FEATURE MATRICES")
print("=" * 80)

X_train_exp3 = hstack([
    X_train_tfidf_original,
    X_train_metadata,
    X_train_article
]).tocsr()

X_test_exp3 = hstack([
    X_test_tfidf_original,
    X_test_metadata,
    X_test_article
]).tocsr()

print("\nFeature combination completed successfully.")

print("\nTF-IDF Features:")
print("Training:", X_train_tfidf_original.shape)
print("Testing :", X_test_tfidf_original.shape)

print("\nMetadata Features:")
print("Training:", X_train_metadata.shape)
print("Testing :", X_test_metadata.shape)

print("\nArticle Reference Feature:")
print("Training:", X_train_article.shape)
print("Testing :", X_test_article.shape)

print("\nFinal Experiment 3 Features:")
print("Training:", X_train_exp3.shape)
print("Testing :", X_test_exp3.shape)

print("\nTotal Features:", X_train_exp3.shape[1])

CREATING EXPERIMENT 3 FEATURE MATRICES

Feature combination completed successfully.

TF-IDF Features:
Training: (33735, 50000)
Testing : (8434, 50000)

Metadata Features:
Training: (33735, 14)
Testing : (8434, 14)

Article Reference Feature:
Training: (33735, 1)
Testing : (8434, 1)

Final Experiment 3 Features:
Training: (33735, 50015)
Testing : (8434, 50015)

Total Features: 50015


In [ ]:
# ============================================================
# EXPERIMENT 3 — TF-IDF + CASE METADATA + ARTICLE REFERENCE
# STEP 3C: Train and Evaluate Random Forest
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
import time

print("=" * 80)
print("EXPERIMENT 3: TF-IDF + CASE METADATA + ARTICLE REFERENCE")
print("=" * 80)

rf_exp3 = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

train_start = time.time()

rf_exp3.fit(
    X_train_exp3,
    y_train
)

train_time_exp3 = time.time() - train_start

# ------------------------------------------------------------
# Prediction
# ------------------------------------------------------------

predict_start = time.time()

y_pred_exp3 = rf_exp3.predict(
    X_test_exp3
)

prediction_time_exp3 = time.time() - predict_start

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

accuracy_exp3 = accuracy_score(y_test, y_pred_exp3)
precision_exp3 = precision_score(y_test, y_pred_exp3)
recall_exp3 = recall_score(y_test, y_pred_exp3)
f1_exp3 = f1_score(y_test, y_pred_exp3)

print("\n" + "=" * 80)
print("EXPERIMENT 3 RESULTS")
print("=" * 80)

print(f"\nTraining Time   : {train_time_exp3:.2f} seconds")
print(f"Prediction Time : {prediction_time_exp3:.4f} seconds")

print(f"\nAccuracy        : {accuracy_exp3:.4f}")
print(f"Precision       : {precision_exp3:.4f}")
print(f"Recall          : {recall_exp3:.4f}")
print(f"F1-Score        : {f1_exp3:.4f}")

print("\n" + "-" * 80)
print("CLASSIFICATION REPORT")
print("-" * 80)

print(
    classification_report(
        y_test,
        y_pred_exp3,
        target_names=["Rejected", "Accepted"]
    )
)

EXPERIMENT 3: TF-IDF + CASE METADATA + ARTICLE REFERENCE

EXPERIMENT 3 RESULTS

Training Time   : 409.93 seconds
Prediction Time : 5.3759 seconds

Accuracy        : 0.9132
Precision       : 0.9179
Recall          : 0.8946
F1-Score        : 0.9061

--------------------------------------------------------------------------------
CLASSIFICATION REPORT
--------------------------------------------------------------------------------
              precision    recall  f1-score   support

    Rejected       0.91      0.93      0.92      4486
    Accepted       0.92      0.89      0.91      3948

    accuracy                           0.91      8434
   macro avg       0.91      0.91      0.91      8434
weighted avg       0.91      0.91      0.91      8434



In [ ]:
# ============================================================
# COMPARING EXPERIMENTS 1, 2 AND 3
# CONFUSION MATRICES + MISCLASSIFICATION ANALYSIS
# ============================================================

from sklearn.metrics import confusion_matrix
import pandas as pd

print("=" * 80)
print("MODEL ERROR ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# Create confusion matrices
# ------------------------------------------------------------

cm_exp1 = confusion_matrix(y_test, y_pred_original_rf)
cm_exp2 = confusion_matrix(y_test, y_pred_exp2)
cm_exp3 = confusion_matrix(y_test, y_pred_exp3)

# ------------------------------------------------------------
# Extract TN, FP, FN, TP
# ------------------------------------------------------------

def get_confusion_metrics(cm):
    tn, fp, fn, tp = cm.ravel()

    total = tn + fp + fn + tp
    correct = tn + tp
    errors = fp + fn
    error_rate = errors / total

    return {
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Correct": correct,
        "Misclassified": errors,
        "Error Rate": error_rate
    }

metrics_exp1 = get_confusion_metrics(cm_exp1)
metrics_exp2 = get_confusion_metrics(cm_exp2)
metrics_exp3 = get_confusion_metrics(cm_exp3)

# ------------------------------------------------------------
# Print individual confusion matrices
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("EXPERIMENT 1 — TF-IDF ONLY")
print("-" * 80)

print("\nConfusion Matrix:")
print(cm_exp1)

print("\nTN:", metrics_exp1["TN"])
print("FP:", metrics_exp1["FP"])
print("FN:", metrics_exp1["FN"])
print("TP:", metrics_exp1["TP"])

print("\nCorrect Predictions :", metrics_exp1["Correct"])
print("Misclassified Cases :", metrics_exp1["Misclassified"])
print(f"Error Rate          : {metrics_exp1['Error Rate']:.4f}")

# ------------------------------------------------------------

print("\n" + "-" * 80)
print("EXPERIMENT 2 — TF-IDF + CASE METADATA")
print("-" * 80)

print("\nConfusion Matrix:")
print(cm_exp2)

print("\nTN:", metrics_exp2["TN"])
print("FP:", metrics_exp2["FP"])
print("FN:", metrics_exp2["FN"])
print("TP:", metrics_exp2["TP"])

print("\nCorrect Predictions :", metrics_exp2["Correct"])
print("Misclassified Cases :", metrics_exp2["Misclassified"])
print(f"Error Rate          : {metrics_exp2['Error Rate']:.4f}")

# ------------------------------------------------------------

print("\n" + "-" * 80)
print("EXPERIMENT 3 — TF-IDF + METADATA + ARTICLE REFERENCE")
print("-" * 80)

print("\nConfusion Matrix:")
print(cm_exp3)

print("\nTN:", metrics_exp3["TN"])
print("FP:", metrics_exp3["FP"])
print("FN:", metrics_exp3["FN"])
print("TP:", metrics_exp3["TP"])

print("\nCorrect Predictions :", metrics_exp3["Correct"])
print("Misclassified Cases :", metrics_exp3["Misclassified"])
print(f"Error Rate          : {metrics_exp3['Error Rate']:.4f}")

# ------------------------------------------------------------
# Overall comparison table
# ------------------------------------------------------------

comparison = pd.DataFrame([
    {
        "Experiment": "Experiment 1",
        "Features": "TF-IDF",
        **metrics_exp1
    },
    {
        "Experiment": "Experiment 2",
        "Features": "TF-IDF + Metadata",
        **metrics_exp2
    },
    {
        "Experiment": "Experiment 3",
        "Features": "TF-IDF + Metadata + Article",
        **metrics_exp3
    }
])

print("\n" + "=" * 80)
print("OVERALL ERROR COMPARISON")
print("=" * 80)

print(
    comparison.to_string(
        index=False,
        formatters={
            "Error Rate": "{:.4f}".format
        }
    )
)

MODEL ERROR ANALYSIS

--------------------------------------------------------------------------------
EXPERIMENT 1 — TF-IDF ONLY
--------------------------------------------------------------------------------

Confusion Matrix:
[[4190  296]
 [ 436 3512]]

TN: 4190
FP: 296
FN: 436
TP: 3512

Correct Predictions : 7702
Misclassified Cases : 732
Error Rate          : 0.0868

--------------------------------------------------------------------------------
EXPERIMENT 2 — TF-IDF + CASE METADATA
--------------------------------------------------------------------------------

Confusion Matrix:
[[4208  278]
 [ 447 3501]]

TN: 4208
FP: 278
FN: 447
TP: 3501

Correct Predictions : 7709
Misclassified Cases : 725
Error Rate          : 0.0860

--------------------------------------------------------------------------------
EXPERIMENT 3 — TF-IDF + METADATA + ARTICLE REFERENCE
--------------------------------------------------------------------------------

Confusion Matrix:
[[4170  316]
 [ 416 3532]

In [ ]:
# ============================================================
# QUALITATIVE ERROR ANALYSIS
# STEP 4A: Find Prediction Changes
# ============================================================

import pandas as pd

print("=" * 80)
print("COMPARING PREDICTION CHANGES")
print("=" * 80)

# Create comparison dataframe
error_analysis = test_df[
    ["case_category", "case_type", "cleaned_judgement", "label"]
].copy()

error_analysis["exp1_prediction"] = y_pred_original_rf
error_analysis["exp2_prediction"] = y_pred_exp2
error_analysis["exp3_prediction"] = y_pred_exp3

# ------------------------------------------------------------
# Correct / incorrect status
# ------------------------------------------------------------

error_analysis["exp1_correct"] = (
    error_analysis["exp1_prediction"] == error_analysis["label"]
)

error_analysis["exp2_correct"] = (
    error_analysis["exp2_prediction"] == error_analysis["label"]
)

error_analysis["exp3_correct"] = (
    error_analysis["exp3_prediction"] == error_analysis["label"]
)

# ------------------------------------------------------------
# Important prediction-change groups
# ------------------------------------------------------------

# Experiment 1 wrong → Experiment 2 correct
exp1_wrong_exp2_correct = error_analysis[
    (~error_analysis["exp1_correct"]) &
    (error_analysis["exp2_correct"])
]

# Experiment 1 correct → Experiment 2 wrong
exp1_correct_exp2_wrong = error_analysis[
    (error_analysis["exp1_correct"]) &
    (~error_analysis["exp2_correct"])
]

# Experiment 2 correct → Experiment 3 wrong
exp2_correct_exp3_wrong = error_analysis[
    (error_analysis["exp2_correct"]) &
    (~error_analysis["exp3_correct"])
]

# Experiment 2 wrong → Experiment 3 correct
exp2_wrong_exp3_correct = error_analysis[
    (~error_analysis["exp2_correct"]) &
    (error_analysis["exp3_correct"])
]

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("EXPERIMENT 1 → EXPERIMENT 2")
print("-" * 80)

print(
    "Experiment 1 WRONG → Experiment 2 CORRECT:",
    len(exp1_wrong_exp2_correct)
)

print(
    "Experiment 1 CORRECT → Experiment 2 WRONG:",
    len(exp1_correct_exp2_wrong)
)

print("\n" + "-" * 80)
print("EXPERIMENT 2 → EXPERIMENT 3")
print("-" * 80)

print(
    "Experiment 2 WRONG → Experiment 3 CORRECT:",
    len(exp2_wrong_exp3_correct)
)

print(
    "Experiment 2 CORRECT → Experiment 3 WRONG:",
    len(exp2_correct_exp3_wrong)
)

# ------------------------------------------------------------
# Prediction-change counts
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("TOTAL PREDICTION CHANGES")
print("-" * 80)

print(
    "Exp 1 → Exp 2 changed:",
    (error_analysis["exp1_prediction"] !=
     error_analysis["exp2_prediction"]).sum()
)

print(
    "Exp 2 → Exp 3 changed:",
    (error_analysis["exp2_prediction"] !=
     error_analysis["exp3_prediction"]).sum()
)

COMPARING PREDICTION CHANGES

--------------------------------------------------------------------------------
EXPERIMENT 1 → EXPERIMENT 2
--------------------------------------------------------------------------------
Experiment 1 WRONG → Experiment 2 CORRECT: 142
Experiment 1 CORRECT → Experiment 2 WRONG: 135

--------------------------------------------------------------------------------
EXPERIMENT 2 → EXPERIMENT 3
--------------------------------------------------------------------------------
Experiment 2 WRONG → Experiment 3 CORRECT: 145
Experiment 2 CORRECT → Experiment 3 WRONG: 152

--------------------------------------------------------------------------------
TOTAL PREDICTION CHANGES
--------------------------------------------------------------------------------
Exp 1 → Exp 2 changed: 277
Exp 2 → Exp 3 changed: 297


In [ ]:
# ============================================================
# QUALITATIVE ERROR ANALYSIS
# STEP 4B: Sample Changed Predictions
# ============================================================

groups = {
    "EXP1 WRONG -> EXP2 CORRECT": exp1_wrong_exp2_correct,
    "EXP1 CORRECT -> EXP2 WRONG": exp1_correct_exp2_wrong,
    "EXP2 WRONG -> EXP3 CORRECT": exp2_wrong_exp3_correct,
    "EXP2 CORRECT -> EXP3 WRONG": exp2_correct_exp3_wrong
}

for group_name, group_df in groups.items():

    print("\n" + "=" * 80)
    print(group_name)
    print("Total cases:", len(group_df))
    print("=" * 80)

    sample = group_df.sample(
        n=min(5, len(group_df)),
        random_state=42
    )

    for idx, row in sample.iterrows():

        print("\n" + "-" * 80)
        print("Test Index:", idx)
        print("Actual Label:", row["label"])
        print("Experiment 1:", row["exp1_prediction"])
        print("Experiment 2:", row["exp2_prediction"])
        print("Experiment 3:", row["exp3_prediction"])
        print("Case Category:", row["case_category"])
        print("Case Type:", row["case_type"])

        print("\nJudgment excerpt:")
        print(row["cleaned_judgement"][:1200])


EXP1 WRONG -> EXP2 CORRECT
Total cases: 142

--------------------------------------------------------------------------------
Test Index: 7596
Actual Label: 0
Experiment 1: 1
Experiment 2: 0
Experiment 3: 1
Case Category: civil
Case Type: writ petition

Judgment excerpt:
arising slp 13067 2019 kerala transport development finance corporation limited appellant versus basil or respondent civil appeal 808 2022 arising slp 13179 2019 civil appeal 807 2022 arising slp 13200 2019 judgment dhananjaya chandrachud leave granted signature not verified batch three appeal arises judgment division bench digitally signed sanjay kumar date 2022 162733 ist reason high court kerala dated april 2019 march 2019 writ appeal judgment single judge dated december 2018 kerala transport development finance corporation limited managing director appellant first two appeal third appeal appellant aneesh babu seek challenge one direction single judge directed seniority list recast placing sherith basil manager sen

In [ ]:
# ============================================================
# FINAL MODEL SELECTION ANALYSIS
# ============================================================

import pandas as pd

print("=" * 80)
print("FINAL MODEL SELECTION ANALYSIS")
print("=" * 80)

final_comparison = pd.DataFrame([
    {
        "Experiment": "Experiment 1",
        "Feature Set": "TF-IDF only",
        "Features": 50000,
        "Accuracy": 0.9132,
        "Precision": 0.9223,
        "Recall": 0.8896,
        "F1-Score": 0.9056,
        "Misclassified": 732
    },
    {
        "Experiment": "Experiment 2",
        "Feature Set": "TF-IDF + Case Metadata",
        "Features": 50014,
        "Accuracy": 0.9140,
        "Precision": 0.9264,
        "Recall": 0.8868,
        "F1-Score": 0.9062,
        "Misclassified": 725
    },
    {
        "Experiment": "Experiment 3",
        "Feature Set": "TF-IDF + Metadata + Article Reference",
        "Features": 50015,
        "Accuracy": 0.9132,
        "Precision": 0.9179,
        "Recall": 0.8946,
        "F1-Score": 0.9061,
        "Misclassified": 732
    }
])

print("\n")
print(final_comparison.to_string(index=False))

# ------------------------------------------------------------
# Improvement relative to Experiment 1
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("IMPROVEMENT OVER BASELINE")
print("=" * 80)

baseline = final_comparison.iloc[0]

for _, row in final_comparison.iloc[1:].iterrows():

    print(f"\n{row['Experiment']}")

    print(
        f"Accuracy change : "
        f"{row['Accuracy'] - baseline['Accuracy']:+.4f}"
    )

    print(
        f"Precision change: "
        f"{row['Precision'] - baseline['Precision']:+.4f}"
    )

    print(
        f"Recall change   : "
        f"{row['Recall'] - baseline['Recall']:+.4f}"
    )

    print(
        f"F1 change       : "
        f"{row['F1-Score'] - baseline['F1-Score']:+.4f}"
    )

    print(
        f"Error change    : "
        f"{row['Misclassified'] - baseline['Misclassified']:+d}"
    )

# ------------------------------------------------------------
# Best model according to each metric
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BEST MODEL BY METRIC")
print("=" * 80)

for metric in ["Accuracy", "Precision", "Recall", "F1-Score"]:
    best_idx = final_comparison[metric].idxmax()
    best = final_comparison.loc[best_idx]

    print(
        f"{metric:<12}: "
        f"{best['Experiment']} ({best[metric]:.4f})"
    )

best_error_idx = final_comparison["Misclassified"].idxmin()
best_error = final_comparison.loc[best_error_idx]

print(
    f"{'Fewest Errors':<12}: "
    f"{best_error['Experiment']} ({best_error['Misclassified']})"
)

FINAL MODEL SELECTION ANALYSIS


  Experiment                           Feature Set  Features  Accuracy  Precision  Recall  F1-Score  Misclassified
Experiment 1                           TF-IDF only     50000    0.9132     0.9223  0.8896    0.9056            732
Experiment 2                TF-IDF + Case Metadata     50014    0.9140     0.9264  0.8868    0.9062            725
Experiment 3 TF-IDF + Metadata + Article Reference     50015    0.9132     0.9179  0.8946    0.9061            732

IMPROVEMENT OVER BASELINE

Experiment 2
Accuracy change : +0.0008
Precision change: +0.0041
Recall change   : -0.0028
F1 change       : +0.0006
Error change    : -7

Experiment 3
Accuracy change : +0.0000
Precision change: -0.0044
Recall change   : +0.0050
F1 change       : +0.0005
Error change    : +0

BEST MODEL BY METRIC
Accuracy    : Experiment 2 (0.9140)
Precision   : Experiment 2 (0.9264)
Recall      : Experiment 3 (0.8946)
F1-Score    : Experiment 2 (0.9062)
Fewest Errors: Experiment 2 (725)


In [ ]:
# ============================================================
# FINAL MODEL — REQUIRED OBJECT CHECK
# ============================================================

print("=" * 80)
print("CHECKING FINAL MODEL OBJECTS")
print("=" * 80)

required_objects = [
    "X_train_tfidf_original",
    "X_test_tfidf_original",
    "X_train_metadata",
    "X_test_metadata",
    "y_train",
    "y_test"
]

for name in required_objects:
    if name in globals():
        obj = globals()[name]
        print(f"✓ {name:<30} {getattr(obj, 'shape', type(obj))}")
    else:
        print(f"✗ {name:<30} NOT AVAILABLE")

print("\nExisting Experiment 2 model:")

if "rf_exp2" in globals():
    print("✓ rf_exp2 available")
    print("  Trees:", rf_exp2.n_estimators)
    print("  Random state:", rf_exp2.random_state)
else:
    print("✗ rf_exp2 NOT AVAILABLE")

CHECKING FINAL MODEL OBJECTS
✓ X_train_tfidf_original         (33735, 50000)
✓ X_test_tfidf_original          (8434, 50000)
✓ X_train_metadata               (33735, 14)
✓ X_test_metadata                (8434, 14)
✓ y_train                        (33735,)
✓ y_test                         (8434,)

Existing Experiment 2 model:
✓ rf_exp2 available
  Trees: 200
  Random state: 42


In [ ]:
# ============================================================
# FINAL MODEL — VALIDATION CHECK
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("=" * 80)
print("FINAL MODEL VALIDATION")
print("=" * 80)

# Predict using the already-trained Experiment 2 model
y_pred_final = rf_exp2.predict(X_test_exp2)

# Calculate metrics
final_accuracy = accuracy_score(y_test, y_pred_final)
final_precision = precision_score(y_test, y_pred_final)
final_recall = recall_score(y_test, y_pred_final)
final_f1 = f1_score(y_test, y_pred_final)

print("\nFinal Model Configuration:")
print("Random Forest")
print("n_estimators :", rf_exp2.n_estimators)
print("random_state :", rf_exp2.random_state)
print("n_jobs       :", rf_exp2.n_jobs)
print("Features     :", X_train_exp2.shape[1])

print("\n" + "-" * 80)
print("FINAL PERFORMANCE")
print("-" * 80)

print(f"\nAccuracy  : {final_accuracy:.4f}")
print(f"Precision : {final_precision:.4f}")
print(f"Recall    : {final_recall:.4f}")
print(f"F1-Score  : {final_f1:.4f}")

print("\n" + "-" * 80)
print("CLASSIFICATION REPORT")
print("-" * 80)

print(
    classification_report(
        y_test,
        y_pred_final,
        target_names=["Rejected", "Accepted"]
    )
)

# Check against our recorded Experiment 2 result
print("\n" + "=" * 80)
print("RECORDED RESULT CHECK")
print("=" * 80)

print("Recorded Accuracy :", 0.9140)
print("Current Accuracy  :", round(final_accuracy, 4))

print("Recorded F1       :", 0.9062)
print("Current F1        :", round(final_f1, 4))

FINAL MODEL VALIDATION

Final Model Configuration:
Random Forest
n_estimators : 200
random_state : 42
n_jobs       : -1
Features     : 50014

--------------------------------------------------------------------------------
FINAL PERFORMANCE
--------------------------------------------------------------------------------

Accuracy  : 0.9140
Precision : 0.9264
Recall    : 0.8868
F1-Score  : 0.9062

--------------------------------------------------------------------------------
CLASSIFICATION REPORT
--------------------------------------------------------------------------------
              precision    recall  f1-score   support

    Rejected       0.90      0.94      0.92      4486
    Accepted       0.93      0.89      0.91      3948

    accuracy                           0.91      8434
   macro avg       0.92      0.91      0.91      8434
weighted avg       0.91      0.91      0.91      8434


RECORDED RESULT CHECK
Recorded Accuracy : 0.914
Current Accuracy  : 0.914
Recorded F1   

In [ ]:
# ============================================================
# FIND FINAL METADATA ENCODER
# ============================================================

print("=" * 80)
print("SEARCHING FOR METADATA ENCODER")
print("=" * 80)

for name, obj in list(globals().items()):

    name_lower = name.lower()

    if any(term in name_lower for term in [
        "encoder",
        "onehot",
        "metadata"
    ]):

        obj_type = type(obj).__name__

        if (
            "encoder" in obj_type.lower()
            or "onehot" in obj_type.lower()
            or "encoder" in name_lower
            or "onehot" in name_lower
        ):
            print(f"\n✓ {name}")
            print("  Type:", type(obj))

SEARCHING FOR METADATA ENCODER

✓ OneHotEncoder
  Type: <class 'type'>

✓ metadata_encoder
  Type: <class 'sklearn.preprocessing._encoders.OneHotEncoder'>


In [ ]:
# ============================================================
# SAVING FINAL MODEL COMPONENTS
# ============================================================

import joblib
import os

save_dir = "/content/final_model"

os.makedirs(save_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. Final Random Forest
# ------------------------------------------------------------

joblib.dump(
    rf_exp2,
    f"{save_dir}/final_random_forest_model.pkl"
)

# ------------------------------------------------------------
# 2. Final TF-IDF Vectorizer
# ------------------------------------------------------------

joblib.dump(
    tfidf_vectorizer,
    f"{save_dir}/final_tfidf_vectorizer.pkl"
)

# ------------------------------------------------------------
# 3. Final Metadata Encoder
# ------------------------------------------------------------

joblib.dump(
    metadata_encoder,
    f"{save_dir}/final_metadata_encoder.pkl"
)

print("=" * 80)
print("FINAL MODEL COMPONENTS SAVED")
print("=" * 80)

print("\nSaved files:")

for filename in os.listdir(save_dir):
    filepath = os.path.join(save_dir, filename)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)

    print(f"✓ {filename}")
    print(f"  Size: {size_mb:.2f} MB")

FINAL MODEL COMPONENTS SAVED

Saved files:
✓ final_random_forest_model.pkl
  Size: 104.71 MB
✓ final_metadata_encoder.pkl
  Size: 0.00 MB
✓ final_tfidf_vectorizer.pkl
  Size: 1.77 MB


In [ ]:
# ============================================================
# SAVING EXPERIMENT 3 BACKUP MODEL
# ============================================================

import joblib
import os

save_dir_exp3 = "/content/experiment_3_backup"

os.makedirs(save_dir_exp3, exist_ok=True)

# ------------------------------------------------------------
# 1. Experiment 3 Random Forest
# ------------------------------------------------------------

joblib.dump(
    rf_exp3,
    f"{save_dir_exp3}/experiment_3_random_forest_model.pkl"
)

# ------------------------------------------------------------
# 2. TF-IDF Vectorizer
# ------------------------------------------------------------

joblib.dump(
    tfidf_vectorizer,
    f"{save_dir_exp3}/experiment_3_tfidf_vectorizer.pkl"
)

# ------------------------------------------------------------
# 3. Metadata Encoder
# ------------------------------------------------------------

joblib.dump(
    metadata_encoder,
    f"{save_dir_exp3}/experiment_3_metadata_encoder.pkl"
)

# ------------------------------------------------------------
# 4. Save Experiment 3 configuration
# ------------------------------------------------------------

experiment_3_config = {
    "model": "Random Forest",
    "n_estimators": 200,
    "random_state": 42,
    "n_jobs": -1,

    "tfidf_features": 50000,
    "metadata_features": 14,
    "article_feature": "has_article_reference",
    "article_feature_count": 1,
    "total_features": 50015,

    "accuracy": 0.9132,
    "precision": 0.9179,
    "recall": 0.8946,
    "f1_score": 0.9061,

    "feature_order": [
        "TF-IDF (50000)",
        "case_category + case_type (14 one-hot features)",
        "has_article_reference (1 binary feature)"
    ]
}

joblib.dump(
    experiment_3_config,
    f"{save_dir_exp3}/experiment_3_config.pkl"
)

# ------------------------------------------------------------
# Display saved files
# ------------------------------------------------------------

print("=" * 80)
print("EXPERIMENT 3 BACKUP SAVED")
print("=" * 80)

for filename in os.listdir(save_dir_exp3):

    filepath = os.path.join(save_dir_exp3, filename)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)

    print(f"\n✓ {filename}")
    print(f"  Size: {size_mb:.2f} MB")

EXPERIMENT 3 BACKUP SAVED

✓ experiment_3_random_forest_model.pkl
  Size: 104.88 MB

✓ experiment_3_config.pkl
  Size: 0.00 MB

✓ experiment_3_tfidf_vectorizer.pkl
  Size: 1.77 MB

✓ experiment_3_metadata_encoder.pkl
  Size: 0.00 MB


In [ ]:
import os

print(os.path.exists("/content/drive/MyDrive"))

if os.path.exists("/content/drive/MyDrive"):
    print("✓ Google Drive is mounted")
else:
    print("✗ Google Drive is NOT mounted")

True
✓ Google Drive is mounted


In [ ]:
import os
import shutil

# Google Drive destination
drive_dir = "/content/drive/MyDrive/Legal_Project/Models"

os.makedirs(drive_dir, exist_ok=True)

# Copy Experiment 3 backup
source_dir = "/content/experiment_3_backup"

for filename in os.listdir(source_dir):
    source = os.path.join(source_dir, filename)
    destination = os.path.join(drive_dir, filename)
    shutil.copy2(source, destination)

print("=" * 80)
print("EXPERIMENT 3 BACKUP COPIED TO GOOGLE DRIVE")
print("=" * 80)

for filename in os.listdir(drive_dir):
    print("✓", filename)

EXPERIMENT 3 BACKUP COPIED TO GOOGLE DRIVE
✓ random_forest_model.pkl
✓ tfidf_vectorizer.pkl
✓ label_mapping.pkl
✓ model_metadata.json
✓ experiment_3_random_forest_model.pkl
✓ experiment_3_config.pkl
✓ experiment_3_tfidf_vectorizer.pkl
✓ experiment_3_metadata_encoder.pkl


In [ ]:
# ============================================================
# LOADING FINAL EXPERIMENT 3 MODEL COMPONENTS
# ============================================================

import os
import joblib

MODEL_DIR = "/content/drive/MyDrive/Legal_Project/Models"

rf_path = os.path.join(
    MODEL_DIR,
    "experiment_3_random_forest_model.pkl"
)

tfidf_path = os.path.join(
    MODEL_DIR,
    "experiment_3_tfidf_vectorizer.pkl"
)

encoder_path = os.path.join(
    MODEL_DIR,
    "experiment_3_metadata_encoder.pkl"
)

config_path = os.path.join(
    MODEL_DIR,
    "experiment_3_config.pkl"
)

print("=" * 80)
print("LOADING FINAL EXPERIMENT 3 MODEL")
print("=" * 80)

final_rf = joblib.load(rf_path)
final_tfidf = joblib.load(tfidf_path)
final_metadata_encoder = joblib.load(encoder_path)
final_config = joblib.load(config_path)

print("\n✓ Random Forest loaded")
print("✓ TF-IDF vectorizer loaded")
print("✓ Metadata encoder loaded")
print("✓ Experiment configuration loaded")

print("\n" + "-" * 80)
print("MODEL CONFIGURATION")
print("-" * 80)

print("Model              :", type(final_rf).__name__)
print("Number of trees    :", final_rf.n_estimators)
print("Random state       :", final_rf.random_state)
print("TF-IDF features    :", len(final_tfidf.get_feature_names_out()))
print("Metadata features  :", len(final_metadata_encoder.get_feature_names_out()))
print("Total features     :", final_rf.n_features_in_)

print("\n" + "-" * 80)
print("EXPECTED")
print("-" * 80)

print("TF-IDF features    : 50000")
print("Metadata features  : 14")
print("Article feature    : 1")
print("Total model input  : 50015")

LOADING FINAL EXPERIMENT 3 MODEL

✓ Random Forest loaded
✓ TF-IDF vectorizer loaded
✓ Metadata encoder loaded
✓ Experiment configuration loaded

--------------------------------------------------------------------------------
MODEL CONFIGURATION
--------------------------------------------------------------------------------
Model              : RandomForestClassifier
Number of trees    : 200
Random state       : 42
TF-IDF features    : 50000
Metadata features  : 14
Total features     : 50015

--------------------------------------------------------------------------------
EXPECTED
--------------------------------------------------------------------------------
TF-IDF features    : 50000
Metadata features  : 14
Article feature    : 1
Total model input  : 50015
